# 소둔로 전체 데이터 시계열 분석

**담당 : 정수진 • issue #58**

**목표:** 7개 CSV를 시간·설비·작업 키로 연결해, 상태 변화의 근거와 예지보전에 사용할 특징 후보를 찾는다. 실제 고장 판정·잔여수명·예측 성능은 확인된 라벨 없이 주장하지 않는다.

**읽는 순서:** 목적(reason) → 연결·계산(how) → 볼 값(what)·시점(when) → 그래프·수치표 → 결과(result)·대안 가설·활용.
수치는 현재 데이터 실행 결과의 기록이다. 원본이나 조건을 바꾸면 마크다운 해석도 갱신한다. 표·그래프는 노트북 안에 표시하고, 그래프는 figures/05_all_data_timeseries_jinny에 함께 저장한다. HTML 보고서는 생성하지 않는다.

| 순서 | 바로가기 | 답하려는 질문 |
|---|---|---|
| 1 | [자료·연결](#sources) | 7개 자료는 어떤 키로 연결되는가? |
| 2 | [설비·부품 기준정보](#masters) | 시간 정보가 없는 마스터를 왜 사용하는가? |
| 3 | [시간·품질](#coverage) | 실제 관측 구간과 결측은 어디인가? |
| 4 | [14 LOT × 22신호](#all-lots) | 모든 신호가 언제 어떻게 달라지는가? |
| 5 | [조업·강종](#events) | 같은 강종 후보에서도 흐름이 다른가? |
| 6 | [센서 간 상관·시차](#correlation) | 어떤 관계가 여러 LOT에서 반복되는가? |
| 7 | [정비](#maintenance) · [부품교체](#replacement) · [정기수리](#plans) | 작업과 관측을 실제로 비교할 수 있는가? |
| 8 | [데이터 간 관계](#cross-data) | 월별 관계·사건 전 관측은 어디까지 믿을 수 있는가? |
| 9 | [파생변수](#features) | 과거 정보만으로 무엇을 계산할 수 있는가? |
| 10 | [통합 시각화·최종 결론](#final) | 예지보전으로 이어지는 근거와 부족한 자료는 무엇인가? |

**범위:** 교육용 재현 자료. 전체 설비의 사건 분포와 CAL01의 센서 관계를 구분한다. 설비마스터·부품마스터는 시계열 측정값이 아닌 기준정보이며, 설치·정비·교체 시각에 연결한다.

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
FIG_DIR = os.path.join("..", "figures")        # 그래프 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

# 기존 3개 (T- 센서 계측, G-02 사건 기록, G-01 기준 정보)
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")          # 소둔로 온도·제어출력, 1초 주기
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")           # 강종변경 등, PLANT_CD 단위
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")       # 계획 정지, PLANT_CD 단위

# 2026-09-28 추가된 4개 (상위 이슈 #43)
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")             # 설비 사전, 키 EQP_TAG
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")            # 부품 사전, 키 PART_CD
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")             # 정비·고장 기록, 키 WO_NO + EQP_TAG
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")        # 부품 교체, 키 WO_NO + PART_CD

ENC = "utf-8-sig"                              # G- 로 시작하는 마스터 CSV는 BOM 있음

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 그래프 한글 깨짐 방지 - 설치된 폰트 중 있는 것을 골라 rcParams에 넣습니다
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False         # 음수 기호가 □ 로 나오는 것 방지
rcParams["figure.dpi"] = 110

<a id="sources"></a>
## 1. 자료와 연결 규칙
**Reason:** 다른 설비의 사건을 온도와 연결하거나 교체 행 수만큼 센서를 복제하면 가짜 관계가 생긴다.
**How:** 센서는 파일명의 CR1-CAL01을 설비마스터의 정확한 태그에 대응시키고, 정비→교체→부품은 `EQP_TAG → WO_NO → PART_CD`로 연결한다.
**What / When:** 키 유일성·미연결·행 증폭과 각 파일의 시간 범위를 확인한다. 라인 이벤트는 CAL01 개별 고장 기록과 구분한다.

| 참고 EDA | 이어서 검증하는 내용 |
|---|---|
| [간격](01_eda_1_sampling_interval.ipynb) · [공백](01_eda_2_time_range_gaps.ipynb) · [LOT](01_eda_3_lot_segments.ipynb) · [결측](01_eda_5_missing_constant.ipynb) | 불규칙 초 간격·LOT 분리·수집 품질 |
| [강종](02_eda_7_steel_grade_temperature_Jinny.ipynb#mapping) · [LOT–이벤트](02_eda_lot_grade_event_alignment.ipynb) | 강종은 시작 일치·구간 내 단일 변경일 때만 후보 연결 |
| [설비](04_equipment_master_timeseries_join_namelesspark.ipynb) · [부품](04_parts_master_yujin.ipynb) | 정적 속성·기준수명과 실제 이력의 구분 |
| [정비](04_eda_g02_maintenance_history_IWEasdwe1234.ipynb#desc-detc-dt) · [교체](03_parts_replacement_history_jinny.ipynb) | 사건 시각·작업 중첩·교체 수량 |

**확인 단위:** 입측 CLN-IN·TC=°C, OP=%, CP-Z4·CP-Z4M=무차원(사용자 확인). 온도 차이·표준편차는 °C, 온도 변화율은 °C/초, OP 차이·표준편차는 %p다. CP 물리적 의미·센서–부품 장착 대응·원본 타임존은 확인되지 않았다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from IPython.display import display
import json
import hashlib

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "data"
fonts = {f.name for f in font_manager.fontManager.ttflist}
for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font in fonts:
        rcParams["font.family"] = font
        break
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

files = {"온도": "T-CR1-CAL01_온도.csv", "이벤트": "G-02_조업이벤트.csv",
         "계획수리": "G-01_정기수리캘린더.csv", "설비": "G-01_설비마스터.csv",
         "부품": "G-01_부품마스터.csv", "정비": "G-02_정비이력.csv", "교체": "G-02_부품교체이력.csv"}
tables = {}
for name, filename in files.items(): tables[name] = pd.read_csv(DATA / filename, encoding="utf-8-sig")
temp = tables["온도"].copy()
event = tables["이벤트"].copy()
plan = tables["계획수리"].copy()
eqp = tables["설비"].copy()
parts = tables["부품"].copy()
maint = tables["정비"].copy()
change = tables["교체"].copy()
TARGET = "P1-CR1-CAL01"
CASE_LOT = 240061  # 화면에서 자세히 볼 LOT. 다른 번호로 바꿔 실행할 수 있습니다.
signals = temp.columns.drop(["MEAS_DT", "LOT_NO"]).tolist()
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (plan, ["STRT_DT", "END_DT"]), (maint, ["DETC_DT", "STRT_DT", "END_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column], errors="raise")
        assert frame[column].notna().all()
assert temp.LOT_NO.notna().all()
assert not temp.duplicated(["LOT_NO", "MEAS_DT"]).any()
for frame, key in [(eqp,"EQP_TAG"), (parts,"PART_CD"), (maint,"WO_NO")]:
    assert frame[key].notna().all() and frame[key].is_unique
assert (maint.STRT_DT <= maint.END_DT).all()
assert change.WO_NO.isin(maint.WO_NO).all()
assert change.PART_CD.isin(parts.PART_CD).all()
assert (eqp.EQP_TAG == TARGET).sum() == 1
cal_maint = maint[maint.EQP_TAG == TARGET].copy()
linked = change.merge(maint, on="WO_NO", validate="m:1").merge(parts, on="PART_CD", validate="m:1")
assert len(linked) == len(change)
cal_parts = linked[linked.EQP_TAG == TARGET].copy()
cr1_event = event[(event.PLANT_CD == "CR1") & (event.EVT_TYPE == "강종변경")].copy()
cr1_plan = plan[plan.PLANT_CD == "CR1"].copy()
temp = temp.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)
temp["dt_s"] = temp.groupby("LOT_NO").MEAS_DT.diff().dt.total_seconds()
temp["segment_id"] = (temp.LOT_NO.ne(temp.LOT_NO.shift()) | temp.dt_s.gt(10)).cumsum()
spans = temp.groupby("LOT_NO").MEAS_DT.agg(start="min", end="max", rows="size")
grade_map = {}
for lot, row in spans.iterrows():
    exact = cr1_event[cr1_event.EVT_DT == row.start]
    inside = cr1_event[cr1_event.EVT_DT.between(row.start, row.end)]
    grade_map[lot] = exact.AFT_VAL.iloc[0] if len(exact) == 1 and len(inside) == 1 else "미확인"
temp["grade_candidate"] = temp.LOT_NO.map(grade_map)


from io import BytesIO
from IPython.display import Image
eqp["INSTALL_DT"] = pd.to_datetime(eqp.INSTALL_DT)
maint = maint.merge(eqp[["EQP_TAG", "PLANT_CD", "EQP_NM"]], on="EQP_TAG", how="left", validate="m:1")
assert maint.PLANT_CD.notna().all()
maint["작업분"] = (maint.END_DT-maint.STRT_DT).dt.total_seconds()/60
maint["대기분"] = (maint.STRT_DT-maint.DETC_DT).dt.total_seconds()/60
cal_maint = maint[maint.EQP_TAG.eq(TARGET)].sort_values("DETC_DT").copy()
linked = change.merge(maint, on="WO_NO", validate="m:1").merge(parts, on="PART_CD", validate="m:1")
cal_parts = linked[linked.EQP_TAG.eq(TARGET)].sort_values("END_DT").copy()
spans["grade"] = spans.index.map(grade_map)
spans["duration_min"] = (spans.end-spans.start).dt.total_seconds()/60
temp["elapsed_min"] = (temp.MEAS_DT-temp.groupby("LOT_NO").MEAS_DT.transform("min")).dt.total_seconds()/60
units = {s: "°C" if s == "CLN-IN" or s.startswith("TC-") else "%" if s.startswith("OP-") else "무차원" if s.startswith("CP-") else "단위 미확인" for s in signals}
LOT_COLORS = {240001:"#577590",240024:"#4D908E",240037:"#D45087",240038:"#9C755F",240044:"#0072B2",240058:"#C44E52",240061:"#E69F00",240069:"#009E73",240091:"#8B5CA6",240108:"#17BECF",240114:"#BCBD22",240117:"#8C564B",240121:"#E377C2",240133:"#7F7F7F"}
SENSOR_COLORS = {"CLN-IN":"#5C677D","TC-Z1":"#7A5195","TC-Z2":"#EF5675","TC-Z3":"#0072B2","TC-Z4":"#009E73","TC-Z5":"#CC79A7","TC-Z6":"#56B4E9","TC-Z7":"#A6761D","TC-Z8":"#66A61E","TC-Z9":"#666666","TC-OUT1":"#D3722C","TC-OUT2":"#8DA0CB","OP-Z1":"#1F77B4","OP-Z2":"#9467BD","OP-Z3":"#E69F00","OP-Z4":"#D55E00","OP-Z5":"#17BECF","OP-Z6":"#BCBD22","OP-Z7":"#8C564B","OP-Z8":"#E377C2","CP-Z4":"#4D908E","CP-Z4M":"#577590"}
SENSOR_COLORS.update({"TC-Z3":"#0072B2","TC-Z4":"#009E73","TC-Z5":"#CC79A7","TC-Z6":"#56B4E9","OP-Z3":"#E69F00","OP-Z4":"#D55E00","CP-Z4":"#4D908E","CP-Z4M":"#577590"})
TYPE_COLORS = {"BM":"#D55E00", "CBM":"#0072B2", "TBM":"#009E73"}
PART_COLORS = {"INS-006":"#9467BD", "SEL-006":"#E69F00", "SEL-009":"#009E73"}
LINE_COLORS = dict(zip(sorted(eqp.PLANT_CD.unique()), sns.color_palette("tab10", eqp.PLANT_CD.nunique()).as_hex()))
WORK_COLORS = {w: "#"+hashlib.sha256(w.encode()).hexdigest()[:6] for w in maint.WO_NO}
WORK_COLORS.update({"WO-2024-01854":"#B07AA1", "WO-2024-01859":"#D55E00"})
PLAN_COLOR = "#777777"
figure_count = 0
def table(frame):
    fmt = lambda x: f"{x:.3e}" if 0 < abs(x) < .001 else f"{x:,.3f}"
    formats = {c: fmt for c in frame.select_dtypes(include="float").columns}
    display(frame.style.format(formats, na_rep="자료 없음"))

FIG05 = ROOT / "figures" / "05_all_data_timeseries_jinny"
FIG05.mkdir(parents=True, exist_ok=True)

def plot(fig):
    global figure_count
    fig.tight_layout()
    image_bytes = BytesIO()
    fig.savefig(image_bytes, format="png", dpi=100, bbox_inches="tight")
    (FIG05 / f"05_{figure_count+1:03d}.png").write_bytes(image_bytes.getvalue())
    display(Image(data=image_bytes.getvalue()))
    plt.close(fig)
    figure_count += 1

def legend_once(ax, **kwargs):
    handles, labels = ax.get_legend_handles_labels()
    pairs = dict(zip(labels, handles))
    ax.legend(pairs.values(), pairs.keys(), **kwargs)

inventory = pd.DataFrame([{"자료":k,"파일":files[k],"행":len(v),"열":len(v.columns),"결측셀":int(v.isna().sum().sum()),"중복행":int(v.duplicated().sum())} for k,v in tables.items()])
table(inventory)
source_hashes = {name:hashlib.sha256((DATA / name).read_bytes()).hexdigest() for name in files.values()}
links = pd.DataFrame([
["온도 → 설비","파일명 → EQP_TAG",len(temp),int((eqp.EQP_TAG==TARGET).sum()),"대상 1행, 시간 이력 아님"],
["정비 → 설비","EQP_TAG",len(maint),int(maint.PLANT_CD.notna().sum()),"전체 정비에서 태그 대조"],
["교체 → 정비 → 부품","WO_NO → PART_CD",len(change),len(linked),"m:1 검증, 교체 행 유지"],
["CAL01 정비","정확한 EQP_TAG",len(cal_maint),cal_maint.WO_NO.nunique(),"작업 건수"],
["CAL01 교체","CAL01 작업의 PART_CD",len(cal_parts),cal_parts.WO_NO.nunique(),"내역 행 / 작업 건수"]
],columns=["연결","키","기준행","연결 대상수","읽는 법"])
table(links)


**Result:** 센서 **345,390행 / 14 LOT / 22신호**, 정비 전체 2,112건 중 CAL01 **44건**, CAL01 교체 **9행·8작업·3품목**이다. 교체는 한 작업에 여러 품목이 올 수 있으므로 작업 수와 행 수가 다르다.

**색 규칙:** LOT별 비교는 LOT 색, 한 LOT의 여러 신호는 센서 색을 사용한다. 같은 신호·LOT·작업은 모든 그림에서 같은 색이다. BM=주홍, CBM=파랑, TBM=초록. 계획수리=회색, 출력 0 구간=노랑. 열지도·산점도의 연속 색은 반드시 색 막대의 수치 기준으로 읽는다. 색이 비슷하더라도 서로 다른 범주의 동일 대상을 뜻하지 않으므로 범례명을 우선한다.

<a id="masters"></a>
## 2. 설비·부품 마스터를 시간에 연결
**Reason:** 온도 변화가 어느 설비의 어떤 점검·부품 이력과 연결되는지 확인하려고 기준정보를 본다.
**How:** CAL01의 설치일→관측기간을 확인하고, 교체에 실제 등장한 부품의 표준수명·조달 리드타임을 대조한다.
**What / When:** 설치일은 단 한 번의 날짜이고 속성 변경 이력은 없다. 부품마스터에는 날짜가 없으므로 아래 부품 그래프는 시간 추세가 아닌 정적 비교다. 교체 시계열은 [7절](#replacement)에서 확인한다.

In [ ]:
target_eqp = eqp[eqp.EQP_TAG.eq(TARGET)].copy()
target_parts = parts[parts.PART_CD.isin(cal_parts.PART_CD)].copy()
table(target_eqp)
table(target_parts)
fig, axes = plt.subplots(1,3, figsize=(16,4))
install = target_eqp.INSTALL_DT.iloc[0]
axes[0].scatter([install],[0], marker="D",color="black",label="CAL01 설치일")
for lot,r in spans.iterrows():
    axes[0].plot([r.start,r.end],[1,1],lw=6,color=LOT_COLORS[lot])
axes[0].set(yticks=[0,1],yticklabels=["설치","14 LOT 관측"],title="설치일과 실제 관측 시점",xlabel="기록 날짜")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
axes[0].legend()
axes[1].barh(target_parts.PART_CD,target_parts.STD_LIFE_H,color=[PART_COLORS[p] for p in target_parts.PART_CD])
axes[1].set(title="CAL01 교체 품목의 표준수명",xlabel="마스터 표준수명 [시간]",ylabel="부품 코드")
axes[2].barh(target_parts.PART_CD,target_parts.LEAD_TIME_D,color=[PART_COLORS[p] for p in target_parts.PART_CD])
axes[2].set(title="CAL01 교체 품목의 조달 리드타임",xlabel="마스터 리드타임 [일]",ylabel="부품 코드")
plot(fig)
master_check = pd.DataFrame([["설비 키",eqp.EQP_TAG.nunique(),len(eqp)],["부품 키",parts.PART_CD.nunique(),len(parts)],["센서 설치 후 관측",(temp.MEAS_DT>=install).sum(),len(temp)]],columns=["검사","충족 수","분모"])
table(master_check)

**Result:** CAL01 설치일은 **2014-05-31**, 중요도 **A**, CBM 대상·데이터 보유 표시는 **Y / Y**다. 모든 센서 행이 설치 이후이지만, 이 사실은 정상 운전을 뜻하지 않는다.

| 실제 교체 품목 | 표준수명 | 리드타임 | 사용 이유 |
|---|---:|---:|---|
| INS-006 가스 분석계 셀 | 9,500시간 | 31일 | 사전 점검 후 조달 의사결정의 기준정보 |
| SEL-006 오일씰 | 8,000시간 | 16일 | 교체 기록을 특정 품목으로 묶기 |
| SEL-009 O링 세트 | 4,500시간 | 5일 | 품목별 교체 간격과 기록 수 비교 |

**가설 판단:** ‘마스터만으로 열화를 예측한다’는 근거가 없다. 한 설비의 속성은 분석 기간에 고정이며 표준수명은 실제 사용·고장 수명이 아니다. **채택한 방향:** 마스터는 정확한 연결·점검 우선순위·조달 검토에 사용하고, RUL 정답이나 센서 상관계수로 사용하지 않는다.

<a id="coverage"></a>
## 3. 시간 지도와 관측 품질
**Reason:** 관측이 없는 시간을 정상으로 채우거나 서로 다른 LOT를 이어 붙이면 추세와 상관이 왜곡된다.
**How:** 원본 순서의 역순·중복을 검사하고, LOT 안에서 실제 초 간격을 계산한다. **10초 초과 공백**에서 분리한다(탐색 설정, 고장 기준 아님).
**What / When:** 연간 지도는 실제 날짜, 결측 지도는 기록된 행 중 결측 비율[%]이다. 빈 시간은 미관측이지 정지가 아니다.

In [ ]:
raw = tables["온도"]
raw_time = pd.to_datetime(raw.MEAS_DT)
reverse_count = int(raw_time.groupby(raw.LOT_NO).diff().dt.total_seconds().lt(0).sum())
intervals = temp.dt_s.dropna().value_counts().sort_index().rename_axis("간격 [초]").reset_index(name="간격 수")
intervals["비율 [%]"] = intervals["간격 수"]/intervals["간격 수"].sum()*100
table(intervals)
table(spans.reset_index().rename(columns={"start":"시작","end":"끝","rows":"관측행","grade":"강종후보","duration_min":"관측범위 [분]"}))
fig,ax = plt.subplots(figsize=(15,5))
for i,(lot,r) in enumerate(spans.iterrows()):
    for _,s in temp[temp.LOT_NO.eq(lot)].groupby("segment_id"):
        ax.plot([s.MEAS_DT.min(),s.MEAS_DT.max()],[i,i],lw=7,color=LOT_COLORS[lot])
for _,p in cr1_plan.iterrows():
    ax.axvspan(p.STRT_DT,p.END_DT+pd.Timedelta(days=1),color=PLAN_COLOR,alpha=.2,label="CR1 계획수리(종료일 포함 가정)")
ax.set(yticks=range(len(spans)),yticklabels=spans.index.astype(str),xlabel="2024년 기록 날짜",ylabel="LOT 번호",title="14 LOT 실제 관측 구간 / 회색은 계획수리")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
legend_once(ax,fontsize=9)
plot(fig)
rate = temp.groupby("segment_id")[signals].diff().div(temp.dt_s,axis=0)
sensor_rows=[]
for lot,rows in temp.groupby("LOT_NO"):
    first = rows.elapsed_min.lt(10)
    last = rows.MEAS_DT.gt(rows.MEAS_DT.max()-pd.Timedelta(minutes=10))
    for s in signals:
        v=rows[s]
        sensor_rows.append({"LOT":lot,"강종후보":grade_map[lot],"센서":s,"단위":units[s],"유효행":v.count(),"결측_%":v.isna().mean()*100,
            "평균":v.mean(),"최소":v.min(),"최대":v.max(),"표준편차":v.std(),"첫10분":v[first].mean(),"끝10분":v[last].mean(),
            "끝-첫":v[last].mean()-v[first].mean(),"첫유효행":v[first].count(),"끝유효행":v[last].count(),
            "변화율절댓값_p99":rate.loc[rows.index,s].abs().quantile(.99)})
sensor_stats = pd.DataFrame(sensor_rows)
missing = sensor_stats.pivot(index="센서",columns="LOT",values="결측_%").reindex(signals)
fig,ax = plt.subplots(figsize=(14,8))
sns.heatmap(missing,annot=True,fmt=".2f",cmap="YlOrRd",vmin=0,ax=ax,cbar_kws={"label":"기록행 중 결측률 [%]"},annot_kws={"size":7})
ax.set(title="전체 22신호 × 14 LOT의 결측률",xlabel="LOT 번호",ylabel="센서")
plot(fig)
table(sensor_stats.loc[sensor_stats["결측_%"].gt(0),["LOT","센서","유효행","결측_%","첫유효행","끝유효행"]].sort_values("결측_%",ascending=False))


**Result:** 원본 LOT 내부 역순 간격 **0건**, 전체 **17연속구간**으로 분리됐다. 1초 간격은 **96.99%**이며 2초 이상도 있으므로 변화량을 실제 초로 나눈다.

**가설 판단:** ‘1년치라 연중 연속 분석 가능’은 관측 범위와 맞지 않는다. ‘전체 결측률이 낮으니 영향이 없다’도 성립하지 않는다. 결측 지도에서 **OP-Z3**처럼 특정 LOT에 모인 결측을 먼저 확인한다. **활용:** LOT·공백을 넘지 않는 이동 창과 유효 관측 비율을 특징에 함께 둔다.

<a id="all-lots"></a>
## 4. 모든 LOT의 모든 22신호
**Reason:** 특정 LOT·Z3만 고르면 다른 존·CP·출측 변화가 가려진다.
**How:** 각 연속구간 안의 30초 평균을 표시하고 통계는 원본에서 계산한다. 공백을 가로질러 선을 잇지 않는다.
**What / When:** 각 LOT의 시작부터 경과분. 센서는 **입측/전단 TC → Z3~Z6 → 후단/출측 TC → OP → CP** 순서로 읽는다. CP-Z4M은 작은 수치이므로 별도 축의 과학적 표기까지 읽는다.
**주의:** 30초 평균은 짧은 급변을 약화시킬 수 있다. 아래 원본 최솟값·최댓값·변화율 표를 함께 확인한다. TC 이외 서로 다른 단위를 크기로 비교하지 않는다.

In [ ]:
families = [
(["CLN-IN"],"입측 CLN-IN","°C"),
(["TC-Z1","TC-Z2"],"전단 Z1~Z2","°C"),
(["TC-Z3","TC-Z4","TC-Z5","TC-Z6"],"중간 Z3~Z6","°C"),
(["TC-Z7","TC-Z8","TC-Z9","TC-OUT1","TC-OUT2"],"후단·출측","°C"),
(["OP-Z1","OP-Z2","OP-Z3","OP-Z4"],"출력 Z1~Z4","%"),
(["OP-Z5","OP-Z6","OP-Z7","OP-Z8"],"출력 Z5~Z8","%"),
(["CP-Z4"],"CP-Z4","무차원"),
(["CP-Z4M"],"CP-Z4M","무차원")]
assert {s for group,_,_ in families for s in group} == set(signals)
display_chunks=[]
for seg, rows in temp.groupby("segment_id"):
    sample=rows.set_index("MEAS_DT")[signals].resample("30s",origin=rows.MEAS_DT.min()).mean()
    sample["LOT_NO"]=rows.LOT_NO.iloc[0]
    sample["segment_id"]=seg
    display_chunks.append(sample.reset_index())
display_data=pd.concat(display_chunks,ignore_index=True)


<a id="lot-240001"></a>
### LOT 240001 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240001
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-01-02 03:36:00 ~ 2024-01-02 15:04:30 / **688.5분**, 40,402행, 강종 후보 **SGCC**.

**Result:** TC-Z3 첫 10분 **860.317°C → 끝 10분 859.541°C**, 차이 **-0.776°C**. OP-Z3는 **79.705% → 80.005%**, 차이 **+0.300%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-OUT1 / -10.433°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240024"></a>
### LOT 240024 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240024
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-03-07 21:22:00 ~ 2024-03-08 06:29:48 / **547.8분**, 31,729행, 강종 후보 **SPFH590**.

**Result:** TC-Z3 첫 10분 **860.208°C → 끝 10분 860.715°C**, 차이 **+0.507°C**. OP-Z3는 **87.397% → 85.593%**, 차이 **-1.804%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-OUT1 / +21.998°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240037"></a>
### LOT 240037 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240037
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-04-13 23:16:00 ~ 2024-04-14 09:27:19 / **611.3분**, 35,348행, 강종 후보 **SPCE**.

**Result:** TC-Z3 첫 10분 **861.497°C → 끝 10분 860.326°C**, 차이 **-1.171°C**. OP-Z3는 **76.498% → 88.321%**, 차이 **+11.823%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-Z9 / +3.219°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240038"></a>
### LOT 240038 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240038
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-04-16 11:35:00 ~ 2024-04-16 13:39:56 / **124.9분**, 7,342행, 강종 후보 **SPCEN**.

**Result:** TC-Z3 첫 10분 **861.825°C → 끝 10분 862.192°C**, 차이 **+0.367°C**. OP-Z3는 **89.365% → 83.176%**, 차이 **-6.189%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-OUT1 / -15.337°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240044"></a>
### LOT 240044 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240044
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-05-03 18:50:00 ~ 2024-05-03 20:04:38 / **74.6분**, 4,334행, 강종 후보 **SPHC**.

**Result:** TC-Z3 첫 10분 **858.976°C → 끝 10분 857.878°C**, 차이 **-1.098°C**. OP-Z3는 **74.773% → 73.030%**, 차이 **-1.743%p**, 전체 결측 **1.731%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-OUT2 / -12.201°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240058"></a>
### LOT 240058 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240058
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-06-09 05:26:00 ~ 2024-06-09 09:02:53 / **216.9분**, 12,559행, 강종 후보 **DP590**.

**Result:** TC-Z3 첫 10분 **860.562°C → 끝 10분 820.479°C**, 차이 **-40.083°C**. OP-Z3는 **86.653% → 20.927%**, 차이 **-65.726%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-Z3 / -40.083°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240061"></a>
### LOT 240061 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240061
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-06-15 01:04:00 ~ 2024-06-15 04:49:23 / **225.4분**, 13,299행, 강종 후보 **SPHC**.

**Result:** TC-Z3 첫 10분 **820.198°C → 끝 10분 857.505°C**, 차이 **+37.307°C**. OP-Z3는 **73.276% → 68.832%**, 차이 **-4.444%p**, 전체 결측 **1.346%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-Z3 / +37.307°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240069"></a>
### LOT 240069 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240069
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-07-07 16:57:00 ~ 2024-07-08 06:29:34 / **812.6분**, 46,836행, 강종 후보 **SPHC**.

**Result:** TC-Z3 첫 10분 **857.797°C → 끝 10분 858.199°C**, 차이 **+0.401°C**. OP-Z3는 **64.232% → 70.250%**, 차이 **+6.018%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-OUT1 / -14.997°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240091"></a>
### LOT 240091 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240091
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-09-08 22:03:00 ~ 2024-09-09 08:44:12 / **641.2분**, 37,633행, 강종 후보 **SPHC**.

**Result:** TC-Z3 첫 10분 **859.592°C → 끝 10분 857.725°C**, 차이 **-1.867°C**. OP-Z3는 **78.060% → 83.749%**, 차이 **+5.689%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-Z9 / -8.255°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240108"></a>
### LOT 240108 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240108
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-10-18 04:05:00 ~ 2024-10-18 11:50:53 / **465.9분**, 26,982행, 강종 후보 **SPFH590**.

**Result:** TC-Z3 첫 10분 **861.206°C → 끝 10분 860.053°C**, 차이 **-1.152°C**. OP-Z3는 **87.070% → 89.626%**, 차이 **+2.556%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-Z9 / +22.045°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240114"></a>
### LOT 240114 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240114
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-11-04 15:40:00 ~ 2024-11-04 20:59:02 / **319.0분**, 18,763행, 강종 후보 **SPCEN**.

**Result:** TC-Z3 첫 10분 **855.723°C → 끝 10분 819.020°C**, 차이 **-36.703°C**. OP-Z3는 **94.198% → 30.172%**, 차이 **-64.026%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-Z4 / -40.420°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240117"></a>
### LOT 240117 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240117
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-11-13 07:08:00 ~ 2024-11-13 14:32:19 / **444.3분**, 25,708행, 강종 후보 **SPCD**.

**Result:** TC-Z3 첫 10분 **822.399°C → 끝 10분 859.316°C**, 차이 **+36.917°C**. OP-Z3는 **82.886% → 80.189%**, 차이 **-2.697%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-Z6 / +37.077°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240121"></a>
### LOT 240121 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240121
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-11-24 07:31:00 ~ 2024-11-24 18:22:12 / **651.2분**, 38,237행, 강종 후보 **SPCE**.

**Result:** TC-Z3 첫 10분 **861.143°C → 끝 10분 860.579°C**, 차이 **-0.564°C**. OP-Z3는 **78.639% → 85.126%**, 차이 **+6.488%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-OUT2 / +12.663°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="lot-240133"></a>
### LOT 240133 · 22신호 상세
**목적:** 이 LOT에서 온도·출력·CP 변화가 같은 시점에 나타나는지 확인한다. **방법:** 아래 8패널을 세로로 비교하고, 원본 통계표에서 첫·끝 10분 변화와 결측을 확인한다. 같은 센서는 다른 LOT에서도 같은 색이다.

In [ ]:
lot = 240133
lot_span=spans.loc[lot]
lot_stats=sensor_stats[sensor_stats.LOT.eq(lot)].copy()
fig,axes=plt.subplots(8,1,figsize=(14,17),sharex=True)
for _,r in display_data[display_data.LOT_NO.eq(lot)].groupby("segment_id"):
    x=(r.MEAS_DT-lot_span.start).dt.total_seconds()/60
    for ax,(group,title,unit) in zip(axes,families):
        for s in group:
            ax.plot(x,r[s],color=SENSOR_COLORS[s],lw=.9,label=s)
        ax.set_ylabel(title+"\n["+unit+"]")
for ax in axes:
    legend_once(ax,ncol=5,fontsize=8)
axes[0].set_title(f"LOT {lot} / {lot_span.grade} / {lot_span.start} ~ {lot_span.end}",color=LOT_COLORS[lot])
axes[-1].set_xlabel("LOT 관측 시작 후 경과시간 [분]")
plot(fig)
table(lot_stats[["센서","단위","유효행","결측_%","최소","최대","첫10분","끝10분","끝-첫","첫유효행","끝유효행"]])
lot_z3=lot_stats.set_index("센서").loc["TC-Z3"]
lot_op=lot_stats.set_index("센서").loc["OP-Z3"]
tc_changes=lot_stats[lot_stats.센서.str.startswith("TC-")].copy()
largest_tc=tc_changes.loc[tc_changes["끝-첫"].abs().idxmax()]


**When:** 2024-12-28 14:57:00 ~ 2024-12-28 16:44:24 / **107.4분**, 6,218행, 강종 후보 **SPCE**.

**Result:** TC-Z3 첫 10분 **860.468°C → 끝 10분 859.668°C**, 차이 **-0.800°C**. OP-Z3는 **80.544% → 84.738%**, 차이 **+4.194%p**, 전체 결측 **0.000%**.

**먼저 볼 지점:** 이 LOT의 TC 신호 중 초후반 차이 절댓값이 가장 큰 것은 **TC-Z9 / +11.902°C**다. 해당 패널을 OP·CP 패널의 같은 경과분과 대조한다. 이 순위는 점검 위치 선정용이며 센서별 정상 범위가 다른 점을 고려해야 한다.

**가설·결론:** 같은 시점의 동반 변화는 공동 운전조건·제어 반응·설비 상태 변화의 후보 근거다. 시작과 끝 평균만으로 전체가 단조 상승·하강했다고 말할 수 없으며 중간 흔들림은 선 그래프에서 확인한다. **활용:** 이 LOT의 평균·변동성·실제 초당 변화율을 다음 LOT와 비교한다. 목표온도·모드가 없어 정상/고장 판정은 보류한다.

<a id="events"></a>
## 5. 조업이벤트와 같은 강종 후보의 시간 흐름
**Reason:** 조업조건 차이를 설비 이상으로 오해하지 않으려면 강종 변경 기록을 먼저 연결해야 한다.
**How:** 전체 라인 월별 이벤트를 집계하고 CR1은 전후 10분 관측 여부를 검사한다. LOT 시작시각 일치·구간 내 단일 변경일 때 AFT_VAL을 강종 후보로 연결한다.
**What / When:** 월별 막대는 기록 건수, LOT 비교는 관측 시작 후 경과분. 생산량·가동시간으로 보정한 빈도가 아니다.

In [ ]:
months=pd.period_range(temp.MEAS_DT.min().to_period("M"),temp.MEAS_DT.max().to_period("M"),freq="M")
event_month=pd.crosstab(event.EVT_DT.dt.to_period("M"),event.PLANT_CD).reindex(months,fill_value=0)
fig,axes=plt.subplots(1,2,figsize=(15,4.5))
event_month.set_axis(event_month.index.astype(str)).plot.bar(ax=axes[0],stacked=True,color=[LINE_COLORS[c] for c in event_month])
axes[0].set(title="전체 라인 조업이벤트 월별 기록",xlabel="이벤트 발생 월",ylabel="이벤트 기록 [건]")
axes[0].legend(title="라인 코드",fontsize=8)
grade_counts=spans.groupby("grade").size().sort_values(ascending=False)
axes[1].bar(grade_counts.index,grade_counts.values,color="#577590")
axes[1].set(title="강종 후보별 관측 LOT 수",xlabel="강종 후보",ylabel="관측 LOT [개]")
plot(fig)
table(event.groupby(["PLANT_CD","EVT_TYPE"]).size().reset_index(name="기록 건수"))
event_rows=[]
for _,r in cr1_event.iterrows():
    pre=temp.MEAS_DT.between(r.EVT_DT-pd.Timedelta(minutes=10),r.EVT_DT,inclusive="left")
    post=temp.MEAS_DT.between(r.EVT_DT,r.EVT_DT+pd.Timedelta(minutes=10),inclusive="left")
    event_rows.append({"EVT_ID":r.EVT_ID,"시각":r.EVT_DT,"전강종":r.BEF_VAL,"후강종":r.AFT_VAL,"전10분행":pre.sum(),"후10분행":post.sum()})
event_coverage=pd.DataFrame(event_rows)
table(event_coverage[event_coverage[["전10분행","후10분행"]].sum(axis=1).gt(0)])
sphc=spans.index[spans.grade.eq("SPHC")].tolist()
fig,axes=plt.subplots(2,1,figsize=(14,7),sharex=True)
for lot in sphc:
    r=display_data[display_data.LOT_NO.eq(lot)]
    for _,seg in r.groupby("segment_id"):
        x=(seg.MEAS_DT-spans.loc[lot,"start"]).dt.total_seconds()/60
        axes[0].plot(x,seg["TC-Z3"],color=LOT_COLORS[lot],label=str(lot))
        axes[1].plot(x,seg["OP-Z3"],color=LOT_COLORS[lot],label=str(lot))
for ax in axes: legend_once(ax,title="LOT 번호",ncol=4)
axes[0].set(title="같은 SPHC 후보: LOT마다 시작 상태와 관측 길이가 다르다",ylabel="TC-Z3 [°C]")
axes[1].set(xlabel="LOT 관측 시작 후 [분]",ylabel="OP-Z3 [%]")
plot(fig)
sphc_stats=sensor_stats[sensor_stats.LOT.isin(sphc)&sensor_stats.센서.eq("TC-Z3")]
table(sphc_stats[["LOT","첫10분","끝10분","끝-첫","최소","최대","첫유효행","끝유효행"]])
sphc_case=sphc_stats.set_index("LOT").loc[240061]


**Result:** CR1 강종변경 **134건** 중 전후 10분 모두 센서 행이 있는 건은 **0건**이다. LOT 시작과 변경 기록이 맞아도 변경 전 효과를 비교할 자료가 있다는 뜻은 아니다.

같은 SPHC 후보 **4 LOT** 중 240061은 Z3 **820.198 → 857.505°C**, **+37.307°C** 차이다. **‘SPHC이면 온도 흐름이 같다’는 가정만으로 설명하기 어렵다.** 목표온도·속도·소재 규격·추출 시작 상태가 대안 설명이다. 강종을 정답 라벨로 쓰지 않고 비교 조건으로 남긴다.

<a id="correlation"></a>
## 6. 어떤 센서끼리 상관이 높고 반복되는가?
**Reason:** 연중 자료를 한꺼번에 합친 상관은 LOT 수준 차이나 같은 상승 추세로 커질 수 있다.
**How:** 22신호의 모든 쌍을 대상으로 ① 합친 원본 Pearson ② LOT별 원본 Pearson 중앙값 ③ LOT별 실제 초당 변화율 Pearson 중앙값을 비교한다. 변화율 Spearman도 대조한다.
**What / When:** 같은 시각·같은 LOT·연속구간 안의 유효쌍만 사용한다. 상수 또는 유효쌍 100 미만은 미산출. 색 범위 -1~1, 무차원이며 0 근처는 선형관계가 약함을 뜻한다. **상관이 높다고 고장 원인·시간 선행성이 증명되지는 않는다.**

In [ ]:
raw_corrs=[]; diff_corrs=[]; rank_corrs=[]; pair_counts=[]
for lot,rows in temp.groupby("LOT_NO"):
    raw_corrs.append(rows[signals].corr(min_periods=100))
    d=rate.loc[rows.index,signals]
    diff_corrs.append(d.corr(min_periods=100))
    rank_corrs.append(d.corr(method="spearman",min_periods=100))
    valid=d.notna().astype("int64")
    pair_counts.append(valid.T.dot(valid))
pooled_corr=temp[signals].corr(min_periods=100)
raw_median=pd.DataFrame(np.nanmedian(np.array(raw_corrs),axis=0),index=signals,columns=signals)
diff_median=pd.DataFrame(np.nanmedian(np.array(diff_corrs),axis=0),index=signals,columns=signals)
fig,axes=plt.subplots(1,3,figsize=(22,7))
for ax,frame,title in zip(axes,[pooled_corr,raw_median,diff_median],["전체 원본을 합친 상관","LOT별 원본 상관 중앙값","LOT별 변화율 상관 중앙값"]):
    sns.heatmap(frame,ax=ax,cmap="coolwarm",vmin=-1,vmax=1,center=0,cbar_kws={"label":"Pearson r [무차원]"})
    ax.set(title=title,xlabel="센서",ylabel="센서")
plot(fig)
pair_rows=[]
for i,a in enumerate(signals):
    for b in signals[i+1:]:
        vals=np.array([r.loc[a,b] for r in diff_corrs])
        median=np.nanmedian(vals)
        pair_rows.append({"센서1":a,"센서2":b,"합친원본_r":pooled_corr.loc[a,b],"LOT원본_r중앙":raw_median.loc[a,b],
            "변화율_r중앙":median,"변화율_Spearman중앙":np.nanmedian([r.loc[a,b] for r in rank_corrs]),
            "유효LOT":int(np.isfinite(vals).sum()),"같은부호LOT":int((vals*median>0).sum()),
            "절댓값0.7이상LOT":int((abs(vals)>=.7).sum()),"최소유효쌍":min(r.loc[a,b] for r in pair_counts)})
pair_summary=pd.DataFrame(pair_rows)
pair_summary["관계크기"]=pair_summary["변화율_r중앙"].abs()
pair_summary=pair_summary.sort_values("관계크기",ascending=False)
table(pair_summary.drop(columns="관계크기").head(20))
strong_pairs=pair_summary[(pair_summary.관계크기>=.7)&(pair_summary.같은부호LOT>=11)]
best_pair=pair_summary.iloc[0]


**Result:** 전체 **231쌍**을 비교했다. 변화율 상관 중앙값 절댓값 ≥0.7, 같은 부호 ≥11 LOT라는 **탐색 기준**을 동시에 만족한 쌍은 **7개**다. 가장 큰 쌍은 **OP-Z5 ↔ CP-Z4M**, 중앙 r **-0.960**, 같은 부호 **14 / 14 LOT**이다.

**읽기:** 왼쪽만 진하고 오른쪽이 옅으면 공통 수준·추세가 관계를 크게 보이게 했을 가능성이 있다. 오른쪽도 진하고 LOT별 부호가 반복되면 동반 변화 특징의 근거가 더 강하다. Spearman과 차이가 크면 일부 급변·비선형성도 확인한다.

**가설 판단:** ‘상관이 크면 모든 변수를 넣는다’보다, 같은 정보를 반복하는 쌍과 보완 정보를 주는 쌍을 구분하는 쪽으로 좁힌다. TC–TC는 **존 차이·온도 범위**, TC–OP는 **시간 변화·제어 관계**, CP 관계는 **정의 확인 후 후보**로 연결한다. 많은 쌍을 탐색한 결과이며 유의성 검정·인과 증명이 아니다.

**상관이 강한 쌍을 구체적으로 읽기**

| 관계 | 합친 원본 r | LOT별 변화율 r 중앙값 | 판단 |
|---|---:|---:|---|
| OP-Z5 ↔ CP-Z4M | 0.329 | -0.960 | LOT 구분 전후에 부호가 달라진다. 전체 상관만으로 해석하면 반대 결론이 가능 |
| TC-Z4 ↔ OP-Z4 | 0.161 | -0.956 | 반복되는 동시 반대 변화. 물리적 원인·선행성은 별개 |
| TC-Z5 ↔ CP-Z4M | 1.000 | 0.898 | CP가 TC로부터 계산된 값인지 생성식 확인 필요. 높은 상관만으로 동일 신호라고 단정하지 않음 |
| TC-Z5 ↔ TC-Z6 | 0.996 | 0.111 | 높은 전체 수준 상관이 강한 초단위 동반 변화까지 뜻하지는 않음 |

**선별 이유:** CP 단위는 무차원으로 확인됐지만 의미·생성식은 미확인이다. 따라서 상관 1위라는 이유로 바로 입력하지 않는다. TC·OP 관계는 관측 가능한 상태 특징 후보로, CP는 정의 확인·중복성 검증 대상으로 구분한다.

### 6-1. 실제 산점도와 LOT별 반복성
**Reason:** 상관 하나에 이상치·특정 LOT가 영향을 줬는지 눈으로 확인한다.
**How / What:** 상위 3쌍의 변화율 산점도에 LOT 고정색을 쓰고, 옆 그림에서 모든 LOT의 원자료 기반 r을 본다. 산점도는 표시량을 줄여 100행 간격으로 추출하며 상관계수는 전체 유효쌍으로 계산한다.
**When:** 같은 시각의 변화율이다. 시간 선후관계는 다음 절에서 따로 본다.

In [ ]:
top_pairs=pair_summary.head(3)
fig,axes=plt.subplots(3,2,figsize=(15,12))
for row,(_,p) in enumerate(top_pairs.iterrows()):
    a,b=p.센서1,p.센서2
    for j,(lot,rows) in enumerate(temp.groupby("LOT_NO")):
        points=rate.loc[rows.index,[a,b]].dropna().iloc[::100]
        axes[row,0].scatter(points[a],points[b],s=4,alpha=.35,color=LOT_COLORS[lot],label=str(lot))
        axes[row,1].scatter(j,diff_corrs[j].loc[a,b],color=LOT_COLORS[lot],s=40)
    ua="%p" if units[a]=="%" else units[a]
    ub="%p" if units[b]=="%" else units[b]
    axes[row,0].set(title=a+" ↔ "+b,xlabel=a+" 변화율 ["+ua+"/초]",ylabel=b+" 변화율 ["+ub+"/초]")
    axes[row,1].set(xticks=range(len(spans)),xticklabels=spans.index.astype(str),ylim=(-1.05,1.05),ylabel="변화율 Pearson r [무차원]",title="같은 부호가 여러 LOT에서 반복되는가?")
    axes[row,1].tick_params(axis="x",rotation=90)
    axes[row,1].axhline(0,color="gray",lw=1)
legend_once(axes[0,0],ncol=4,fontsize=7,title="LOT 번호")
plot(fig)
table(top_pairs.drop(columns="관계크기"))


**Result:** 위 표가 산점도의 실제 근거다. 점 구름이 좁은 직선에 가깝고 오른쪽 LOT별 점도 같은 부호이면 관계가 특정 LOT 하나에만 의존하지 않는다. **상관 상위쌍을 고른 사후 시각화**이므로 새로운 데이터의 재현을 보장하지 않는다.

**활용:** 관련 센서를 모두 복제 입력하기보다 차이·범위·변동성을 특징으로 만들고, 이후 시간순 검증에서 TC 단독 대비 결합 특징의 추가 이득을 평가한다.

### 6-2. 8개 존의 출력–온도 시차
**Reason:** 동시 음의 상관을 ‘출력이 온도를 내린다’고 해석하면 제어 피드백을 인과로 오해할 수 있다.
**How:** 같은 세그먼트에서 정확한 타임스탬프를 맞춰 **OP 변화율(t−lag) ↔ TC 변화율(t)**을 계산한다. 행 이동을 초 이동으로 간주하지 않는다.
**What / When:** 양수 lag는 출력이 앞선 관측, 음수는 출력이 뒤인 관측이다. 음수는 관계 진단용이고 모델 입력에 미래 출력을 넣지 않는다. ±120·60·30·10초와 0초를 비교한다.

In [ ]:
lags=[-120,-60,-30,-10,0,10,30,60,120]
lag_rows=[]
for lot,rows in temp.groupby("LOT_NO"):
    for zone in range(1,9):
        tc,op=f"TC-Z{zone}",f"OP-Z{zone}"
        for lag in lags:
            chunks=[]
            for _,seg in rows.groupby("segment_id"):
                times=pd.DatetimeIndex(seg.MEAS_DT)
                tc_rate=rate.loc[seg.index,tc].to_numpy()
                op_rate=pd.Series(rate.loc[seg.index,op].to_numpy(),index=times)
                chunks.append(pd.DataFrame({"TC":tc_rate,"OP":op_rate.reindex(times-pd.Timedelta(seconds=lag)).to_numpy()}))
            pairs=pd.concat(chunks,ignore_index=True).dropna()
            r=pairs.TC.corr(pairs.OP) if len(pairs)>=100 and pairs.nunique().min()>1 else np.nan
            lag_rows.append({"LOT":lot,"존":zone,"lag_초":lag,"유효쌍":len(pairs),"r":r})
lag_results=pd.DataFrame(lag_rows)
lag_median=lag_results.groupby(["존","lag_초"]).r.median().unstack().reindex(columns=lags)
lag_n=lag_results.groupby(["존","lag_초"]).유효쌍.min().unstack().reindex(columns=lags)
fig,axes=plt.subplots(1,2,figsize=(17,6))
sns.heatmap(lag_median,ax=axes[0],annot=True,fmt=".2f",cmap="coolwarm",vmin=-1,vmax=1,cbar_kws={"label":"LOT별 r 중앙값 [무차원]"})
axes[0].set(title="존별 출력–온도 변화율의 시간차 관계",xlabel="lag [초] / +출력 앞섬, -출력 뒤",ylabel="존")
detail=lag_results[lag_results.존.eq(3)].pivot(index="LOT",columns="lag_초",values="r").reindex(columns=lags)
sns.heatmap(detail,ax=axes[1],annot=True,fmt=".2f",cmap="coolwarm",vmin=-1,vmax=1,cbar_kws={"label":"Z3 변화율 r [무차원]"})
axes[1].set(title="Z3: LOT별 시차 관계가 반복되는가?",xlabel="lag [초]",ylabel="LOT")
plot(fig)
table(lag_median)
table(lag_n.rename_axis(index="존 / 최소 유효쌍",columns="lag [초]"))


**Result:** 동시 변화율 상관의 LOT 중앙값은 **Z3 -0.837, Z4 -0.956**이다. 총 **1008 LOT·존·lag 조합**을 확인했다.

**가설 판단:** ‘현재 OP만으로 온도 수준을 설명’하는 것은 부족하다. 0초에서 강한 음의 관계가 보여도 제어기 반응·목표온도·운전 상태가 섞일 수 있다. ±시차별 강도와 LOT 간 일관성을 보고 **관측상의 시간 관계**까지만 말한다.

**활용:** 과거 60·300초의 TC·OP 평균·변동·변화율, 존 간 차이를 후보로 남긴다. 최적 물리 지연을 확정하려면 목표값(SP)·제어 모드·센서 동기화 정보가 더 필요하다.

**시차 결과의 핵심:** 0초를 제외한 ±10~120초에서 존별 중앙 r의 최대 절댓값은 **0.048**다. 이번 비교에서는 **동시 관계는 강하지만 뚜렷한 지연 선행관계는 확인되지 않았다.** 60·300초 특징 창을 고장 예측 선행시간으로 해석하지 않는다.

<a id="maintenance"></a>
## 7. 정비이력: 발생·대기·작업·기록 정지시간
**Reason:** 고장 후보의 시점과 실제로 예측 전에 관측한 센서가 있는지 구분해야 한다.
**How:** 전체 정비는 설비마스터로 라인을 붙이고, 센서 분석은 CAL01로 제한한다. 감지(DETC_DT), 착수(STRT_DT), 완료(END_DT)를 구분한다.
**What / When:** 월별 건수는 감지 월, 대기=착수−감지[분], 작업길이=완료−착수[분], DOWN_MIN은 별도의 기록 정지시간[분]. 완료시각을 재가동 확인시각으로 단정하지 않는다.

In [ ]:
all_mnt_month=pd.crosstab(maint.DETC_DT.dt.to_period("M"),maint.PLANT_CD).reindex(months,fill_value=0)
cal_mnt_month=pd.crosstab(cal_maint.DETC_DT.dt.to_period("M"),cal_maint.MNT_TYPE).reindex(months,fill_value=0).reindex(columns=["BM","CBM","TBM"],fill_value=0)
fig,axes=plt.subplots(2,2,figsize=(16,10))
all_mnt_month.set_axis(all_mnt_month.index.astype(str)).plot.bar(stacked=True,ax=axes[0,0],color=[LINE_COLORS[c] for c in all_mnt_month])
axes[0,0].set(title="전체 설비 정비 / 감지 월",xlabel="월",ylabel="작업 [건]")
axes[0,0].legend(title="라인",ncol=4,fontsize=7)
cal_mnt_month.set_axis(cal_mnt_month.index.astype(str)).plot.bar(stacked=True,ax=axes[0,1],color=[TYPE_COLORS[c] for c in cal_mnt_month])
axes[0,1].set(title="CAL01 정비 / 감지 월",xlabel="월",ylabel="작업 [건]")
for typ,rows in cal_maint.groupby("MNT_TYPE"):
    axes[1,0].scatter(rows.DETC_DT,rows["대기분"]/60,color=TYPE_COLORS[typ],label=typ)
    axes[1,1].scatter(rows["작업분"],rows.DOWN_MIN,color=TYPE_COLORS[typ],label=typ)
axes[1,0].set(title="CAL01 감지→착수 대기시간",xlabel="감지 날짜",ylabel="대기 [시간]")
axes[1,0].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
limit=max(cal_maint["작업분"].max(),cal_maint.DOWN_MIN.max())
axes[1,1].plot([0,limit],[0,limit],"--",color="gray",label="같은 값 참고선")
axes[1,1].set(title="작업길이와 기록 정지시간은 같은가?",xlabel="완료-착수 [분]",ylabel="DOWN_MIN [분]")
axes[1,0].legend(title="정비유형"); axes[1,1].legend(title="정비유형")
plot(fig)
type_stats=cal_maint.groupby("MNT_TYPE").agg(작업수=("WO_NO","size"),고장모드기재=("FAIL_MODE","count"),
    대기시간중앙_분=("대기분","median"),작업길이중앙_분=("작업분","median"),기록정지중앙_분=("DOWN_MIN","median"))
table(type_stats)
table(cal_maint[["WO_NO","DETC_DT","STRT_DT","END_DT","MNT_TYPE","FAIL_MODE","대기분","작업분","DOWN_MIN","PART_CHG"]])


**Result:** CAL01 **44건**의 유형별 건수·시간은 위 표와 같다. 고장모드가 기재된 작업은 **20건**이며, 작업길이와 DOWN_MIN이 일치하지 않는 작업은 **34건**이다.

**가설 판단:** 작업시간 전체를 설비 정지시간으로 대체하지 않는다. 월별 정비 건수는 가동시간·생산량이 없어 고장률이 아니고 BM 코드·고장모드 기재만으로 센서 이상 라벨을 확정하지 않는다. **활용:** 감지 이전을 전조 탐색 시점으로 삼고, 감지·작업 후 생기는 결과 컬럼은 모델 입력에서 분리한다.

### 7-1. 감지 전 관측과 작업 전후 비교 가능성
**Reason:** 사건이 있어도 직전 센서가 없으면 전조를 학습하거나 조기 경보를 평가할 수 없다.
**How:** CAL01 44건 각각에 대해 감지 전 1·6·24시간의 행 수·관측지지시간, 착수 전/완료 후 10분 관측을 계산한다.
**What / When:** 관측지지시간은 같은 연속구간에서 두 유효 기록시각 사이 간격을 더한 보수적 범위이며, 전체 센서값이 유효하다는 뜻은 아니다. 24시간에 몇 행만 있어도 ‘충분한 전조 자료’로 보지 않는다.

In [ ]:
def supported_seconds(start,end):
    inside=temp.MEAS_DT.ge(start)&temp.MEAS_DT.lt(end)&temp.dt_s.le(10)&temp.groupby("LOT_NO").MEAS_DT.diff().notna()
    valid=temp.loc[inside,["MEAS_DT","dt_s"]].copy()
    # 세그먼트 경계의 간격은 제외하고 창 시작 경계를 자릅니다.
    valid=valid[temp.loc[valid.index,"segment_id"].eq(temp.segment_id.shift().loc[valid.index])]
    lengths=np.minimum(valid.dt_s,(valid.MEAS_DT-start).dt.total_seconds())
    return float(lengths.clip(lower=0).sum())

work_rows=[]; before_after=[]
for _,w in cal_maint.iterrows():
    pre=temp[temp.MEAS_DT.between(w.STRT_DT-pd.Timedelta(minutes=10),w.STRT_DT,inclusive="left")]
    post=temp[temp.MEAS_DT.between(w.END_DT,w.END_DT+pd.Timedelta(minutes=10),inclusive="right")]
    overlaps=cal_maint[(cal_maint.WO_NO!=w.WO_NO)&(cal_maint.STRT_DT<w.END_DT+pd.Timedelta(minutes=10))&(cal_maint.END_DT>w.STRT_DT-pd.Timedelta(minutes=10))]
    rec={"WO_NO":w.WO_NO,"감지":w.DETC_DT,"유형":w.MNT_TYPE,"전10분행":len(pre),"후10분행":len(post),"중첩수":len(overlaps),"중첩작업":", ".join(overlaps.WO_NO)}
    for hours in [1,6,24]:
        start=w.DETC_DT-pd.Timedelta(hours=hours)
        rec[f"감지전{hours}h행"]=int(temp.MEAS_DT.between(start,w.DETC_DT,inclusive="left").sum())
        rec[f"감지전{hours}h관측_%"]=supported_seconds(start,w.DETC_DT)/(hours*3600)*100
    work_rows.append(rec)
    if len(pre) and len(post):
        for s in signals:
            before_after.append({"WO_NO":w.WO_NO,"센서":s,"단위":units[s],"전평균":pre[s].mean(),"후평균":post[s].mean(),"후-전":post[s].mean()-pre[s].mean(),
                "전유효행":pre[s].count(),"후유효행":post[s].count(),"전LOT":str(pre.LOT_NO.unique().tolist()),"후LOT":str(post.LOT_NO.unique().tolist()),"중첩수":len(overlaps)})
work_coverage=pd.DataFrame(work_rows)
work_comparison=pd.DataFrame(before_after)
usable_work=work_coverage[work_coverage.전10분행.gt(0)&work_coverage.후10분행.gt(0)]
fig,axes=plt.subplots(1,2,figsize=(17,10))
heat=work_coverage.set_index("WO_NO")[[f"감지전{h}h관측_%" for h in [1,6,24]]]
sns.heatmap(heat,ax=axes[0],annot=True,fmt=".1f",vmin=0,vmax=100,cmap="YlGnBu",cbar_kws={"label":"창 길이 대비 관측지지시간 [%]"},annot_kws={"size":7})
axes[0].set(title="44작업 감지 전에 센서가 얼마나 있었나?",xlabel="감지 이전 시간 창",ylabel="작업번호")
counts=[len(cal_maint),int(work_coverage["감지전24h행"].gt(0).sum()),len(usable_work),int((usable_work.중첩수.eq(0)).sum())]
axes[1].bar(["전체 작업","감지 전24h\n일부 관측","전후10분\n양쪽 관측","전후 관측·\n중첩 없음"],counts,color=["#777777","#0072B2","#E69F00","#009E73"])
for i,v in enumerate(counts): axes[1].text(i,v+.3,str(v),ha="center")
axes[1].set(title="서로 다른 검토 조건별 작업 수",ylabel="작업 [건]")
plot(fig)
table(work_coverage)


**Result:** 정비 **44건** 중 감지 전 24시간에 일부 센서가 있는 것은 **8건**, 시작 전·완료 후 10분 양쪽 관측은 **2건**이다. 양쪽 관측이면서 다른 작업이 중첩되지 않은 건은 **0건**이다.

**가설 판단:** ‘정비 기록을 붙이면 모든 고장의 전조를 학습할 수 있다’는 근거가 부족하다. 일부 행 존재와 충분한 선행시간 확보를 구분해야 한다. **활용:** 위 1·6·24시간 표로 평가 가능한 사건부터 추린 후, 실제 고장 확인·해당 센서의 유효율·운전 조건을 추가 대조한다.

### 7-2. WO-2024-01854의 센서–작업 겹침
**Reason:** 전후 숫자 차이를 정비 효과로 잘못 해석하지 않으려는 비교다.
**How / What:** TC-Z3~Z6·OP-Z3~Z4를 같은 실제 시간축에 두고, 대상 작업은 굵은 테두리, 다른 중첩 작업은 사선으로 표시한다. 작업번호의 색은 다른 그림에서도 유지한다.
**When:** 시작 전 10분부터 완료 후 10분까지. 표는 모든 22신호의 전후 평균과 유효행 수다.

In [ ]:
job="WO-2024-01854"
w=cal_maint.set_index("WO_NO").loc[job]
view=temp[temp.MEAS_DT.between(w.STRT_DT-pd.Timedelta(minutes=10),w.END_DT+pd.Timedelta(minutes=10))]
nearby=cal_maint[(cal_maint.STRT_DT<view.MEAS_DT.max())&(cal_maint.END_DT>view.MEAS_DT.min())]
fig,axes=plt.subplots(2,1,figsize=(14,7),sharex=True)
for _,seg in view.groupby("segment_id"):
    for s in ["TC-Z3","TC-Z4","TC-Z5","TC-Z6"]: axes[0].plot(seg.MEAS_DT,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(seg.MEAS_DT,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
for ax in axes:
    for _,o in nearby.iterrows():
        ax.axvspan(max(o.STRT_DT,view.MEAS_DT.min()),min(o.END_DT,view.MEAS_DT.max()),facecolor=WORK_COLORS[o.WO_NO],edgecolor=WORK_COLORS[o.WO_NO],alpha=.2,linewidth=2.5 if o.WO_NO==job else 1,hatch=None if o.WO_NO==job else "//",label=o.WO_NO)
    legend_once(ax,ncol=3,fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
axes[0].set(title=job+" / 중첩 작업을 함께 확인",ylabel="온도 [°C]")
axes[1].set(xlabel="실제 기록 시각 [월-일 시:분]",ylabel="출력 [%]")
plot(fig)
work_values=work_comparison[work_comparison.WO_NO.eq(job)]
table(work_values)
job_z3=work_values.set_index("센서").loc["TC-Z3"]
job_op=work_values.set_index("센서").loc["OP-Z3"]


**Result:** Z3 전후 평균 차이 **+0.404°C**, OP-Z3 **+13.967%p**. Z3 전후 유효행은 **582 / 581**이며 중첩 작업 **1건**이 있다.

**가설 판단:** ‘작업 전후 값이 달라졌다’는 관측 사실이지만 ‘이 작업 때문에 회복됐다’는 결론은 보류한다. 목표값·동일 운전조건·작업 중첩이 통제되지 않았다. **활용:** 전후 변화가 큰 신호와 해당 작업 상세를 함께 조사하고, 고장 전조 분석에는 감지 이전 값만 사용한다.

### 7-2. WO-2024-01859의 센서–작업 겹침
**Reason:** 전후 숫자 차이를 정비 효과로 잘못 해석하지 않으려는 비교다.
**How / What:** TC-Z3~Z6·OP-Z3~Z4를 같은 실제 시간축에 두고, 대상 작업은 굵은 테두리, 다른 중첩 작업은 사선으로 표시한다. 작업번호의 색은 다른 그림에서도 유지한다.
**When:** 시작 전 10분부터 완료 후 10분까지. 표는 모든 22신호의 전후 평균과 유효행 수다.

In [ ]:
job="WO-2024-01859"
w=cal_maint.set_index("WO_NO").loc[job]
view=temp[temp.MEAS_DT.between(w.STRT_DT-pd.Timedelta(minutes=10),w.END_DT+pd.Timedelta(minutes=10))]
nearby=cal_maint[(cal_maint.STRT_DT<view.MEAS_DT.max())&(cal_maint.END_DT>view.MEAS_DT.min())]
fig,axes=plt.subplots(2,1,figsize=(14,7),sharex=True)
for _,seg in view.groupby("segment_id"):
    for s in ["TC-Z3","TC-Z4","TC-Z5","TC-Z6"]: axes[0].plot(seg.MEAS_DT,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(seg.MEAS_DT,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
for ax in axes:
    for _,o in nearby.iterrows():
        ax.axvspan(max(o.STRT_DT,view.MEAS_DT.min()),min(o.END_DT,view.MEAS_DT.max()),facecolor=WORK_COLORS[o.WO_NO],edgecolor=WORK_COLORS[o.WO_NO],alpha=.2,linewidth=2.5 if o.WO_NO==job else 1,hatch=None if o.WO_NO==job else "//",label=o.WO_NO)
    legend_once(ax,ncol=3,fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
axes[0].set(title=job+" / 중첩 작업을 함께 확인",ylabel="온도 [°C]")
axes[1].set(xlabel="실제 기록 시각 [월-일 시:분]",ylabel="출력 [%]")
plot(fig)
work_values=work_comparison[work_comparison.WO_NO.eq(job)]
table(work_values)
job_z3=work_values.set_index("센서").loc["TC-Z3"]
job_op=work_values.set_index("센서").loc["OP-Z3"]


**Result:** Z3 전후 평균 차이 **+20.838°C**, OP-Z3 **+1.114%p**. Z3 전후 유효행은 **567 / 574**이며 중첩 작업 **1건**이 있다.

**가설 판단:** ‘작업 전후 값이 달라졌다’는 관측 사실이지만 ‘이 작업 때문에 회복됐다’는 결론은 보류한다. 목표값·동일 운전조건·작업 중첩이 통제되지 않았다. **활용:** 전후 변화가 큰 신호와 해당 작업 상세를 함께 조사하고, 고장 전조 분석에는 감지 이전 값만 사용한다.

<a id="replacement"></a>
## 8. 부품교체의 시간 흐름과 부품 기준정보
**Reason:** 교체가 반복되는 품목과 시점을 알아야 센서 변화의 맥락을 좁히고 조달·점검 후보를 정할 수 있다.
**How:** 교체에는 날짜가 없으므로 WO_NO로 정비의 END_DT를 붙인다. 이는 **완료시각을 사용한 교체 시점 대리값**이며 실제 탈착 시각이 아니다.
**What / When:** 전체 월별 교체는 내역 행 수, CAL01 점의 크기는 CSV QTY[기록 수량]이다. 품목별 수량 단위 정의가 없어 ‘개’로 단정하지 않는다.

In [ ]:
replacement_month=pd.crosstab(linked.END_DT.dt.to_period("M"),linked.PLANT_CD).reindex(months,fill_value=0)
cal_parts["동일품목직전완료"]=cal_parts.groupby("PART_CD").END_DT.shift()
cal_parts["이전기록간격_시간"]=(cal_parts.END_DT-cal_parts["동일품목직전완료"]).dt.total_seconds()/3600
cal_parts["간격_표준수명비"]=cal_parts["이전기록간격_시간"]/cal_parts.STD_LIFE_H
has_change=maint.WO_NO.isin(change.WO_NO)
flag_disagree=int((has_change.ne(maint.PART_CHG.eq("Y"))).sum())
fig,axes=plt.subplots(2,1,figsize=(14,8))
replacement_month.set_axis(replacement_month.index.astype(str)).plot.bar(ax=axes[0],stacked=True,color=[LINE_COLORS[c] for c in replacement_month])
axes[0].set(title="전체 라인 부품교체 내역 / 연결된 작업 완료 월",xlabel="월",ylabel="교체 내역 [행]")
axes[0].legend(title="라인",ncol=4,fontsize=8)
part_order=sorted(cal_parts.PART_CD.unique())
for i,p in enumerate(part_order):
    rows=cal_parts[cal_parts.PART_CD.eq(p)]
    axes[1].scatter(rows.END_DT,[i]*len(rows),s=rows.QTY*50,color=PART_COLORS[p],label=p)
    for _,r in rows.iterrows(): axes[1].annotate(str(int(r.QTY)),(r.END_DT,i),xytext=(0,8),textcoords="offset points",ha="center")
axes[1].set(yticks=range(len(part_order)),yticklabels=part_order,xlabel="연결된 작업 완료일",ylabel="부품 코드",title="CAL01 교체 시점 / 점 위 숫자=QTY, 면적∝기록 수량")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d")); axes[1].legend()
plot(fig)
table(cal_parts[["WO_NO","PART_CD","PART_NM","END_DT","QTY","CHG_RSN","MNT_TYPE","STD_LIFE_H","LEAD_TIME_D","동일품목직전완료","이전기록간격_시간","간격_표준수명비"]])
part_summary=cal_parts.groupby(["PART_CD","PART_NM"]).agg(기록행=("WO_NO","size"),작업수=("WO_NO","nunique"),기록수량합=("QTY","sum"),최초=("END_DT","min"),최종=("END_DT","max"),간격계산수=("이전기록간격_시간","count"),간격중앙_시간=("이전기록간격_시간","median"))
table(part_summary)


**Result:** 전체 교체 427행 중 CAL01은 **9행 / 8작업 / 3품목**이다. 정비의 PART_CHG 표시와 실제 교체 내역 존재가 어긋난 작업은 전체 **0건**이다. CAL01에서 동일 품목의 이전 기록 간격을 계산할 수 있는 것은 **6행**이다.

**가설 판단:** ‘교체 간격=부품 수명’은 채택하지 않는다. 같은 품목 코드에 여러 실물·장착 위치가 포함될 수 있고 일부 수량만 교체할 수 있다. 첫 기록 이전 이력도 없다. **간격/표준수명 비율은 기록 대조값이며 수명 소진율·고장 확률·잔여수명이 아니다.**

**활용:** INS-006·SEL-006·SEL-009별 마지막 **기록된 완료 이후 경과시간**을 맥락 후보로 만들고, 장착 위치·실물 식별자·가동시간을 확보하면 실제 사용수명 검증으로 확장한다.

<a id="plans"></a>
## 9. 정기수리캘린더와 실제 관측
**Reason:** 계획 정지를 고장 라벨로 쓰거나 계획 전후의 다른 LOT를 수리 효과로 오해하지 않기 위해 본다.
**How:** 전체 4개 계획을 날짜 구간으로 그리고, CR1 2건은 센서와 겹치는 행 및 가장 가까운 전후 LOT를 찾는다.
**What / When:** 날짜만 있으므로 종료일 다음 날 0시를 끝 경계로 하는 **종료일 포함 가정**이다. 실제 수행·재가동 시각은 이 파일에 없다.

In [ ]:
plan_rows=[]
for _,p in cr1_plan.iterrows():
    end=p.END_DT+pd.Timedelta(days=1)
    pre=spans[spans.end<p.STRT_DT].sort_values("end")
    post=spans[spans.start>=end].sort_values("start")
    before=pre.index[-1] if len(pre) else None
    after=post.index[0] if len(post) else None
    plan_rows.append({"계획":p.PLAN_ID,"시작":p.STRT_DT,"종료경계":end,"기간내센서행":int(temp.MEAS_DT.between(p.STRT_DT,end,inclusive="left").sum()),
        "직전LOT":before,"직전강종":grade_map.get(before),"직후LOT":after,"직후강종":grade_map.get(after),
        "시작전공백_시간":(p.STRT_DT-spans.loc[before,"end"]).total_seconds()/3600 if before else np.nan,
        "종료후공백_시간":(spans.loc[after,"start"]-end).total_seconds()/3600 if after else np.nan,
        "기간내CAL완료작업":int(cal_maint.END_DT.between(p.STRT_DT,end,inclusive="left").sum())})
plan_evidence=pd.DataFrame(plan_rows)
fig,axes=plt.subplots(1,2,figsize=(16,5))
for i,(_,p) in enumerate(plan.iterrows()):
    axes[0].plot([p.STRT_DT,p.END_DT+pd.Timedelta(days=1)],[i,i],lw=12,color=PLAN_COLOR)
    axes[0].text(p.END_DT+pd.Timedelta(days=2),i,p.PLANT_CD,va="center")
axes[0].set(yticks=range(len(plan)),yticklabels=plan.PLAN_ID,xlabel="계획 날짜",title="전체 계획수리: 수행 여부와 구분")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
comparison=[]
for _,p in plan_evidence.iterrows():
    for side,key in [("직전","직전LOT"),("직후","직후LOT")]:
        lot=int(p[key])
        r=sensor_stats[sensor_stats.LOT.eq(lot)&sensor_stats.센서.eq("TC-Z3")].iloc[0]
        comparison.append({"계획":p.계획,"위치":side,"LOT":lot,"강종후보":grade_map[lot],"Z3전체평균_°C":r.평균,"Z3첫10분_°C":r.첫10분,"Z3끝10분_°C":r.끝10분})
        axes[1].scatter(p.계획+"\n"+side,r.평균,color=LOT_COLORS[lot],s=80,label=f"{lot} / {grade_map[lot]}")
axes[1].set(title="계획 전후 가까운 LOT / 조건이 같지 않음",ylabel="TC-Z3 LOT 전체평균 [°C]",xlabel="계획 / 관측 위치")
axes[1].legend(fontsize=8)
plot(fig)
table(plan_evidence)
table(pd.DataFrame(comparison))


**Result:** CR1 계획수리 기간에 센서가 겹치는 행은 **0행**이다. 6월은 **240058(DP590) → 240061(SPHC)**, 11월도 표와 같이 전후 LOT·강종 후보가 다르다.

**가설 판단:** ‘계획수리 후 온도가 달라졌으니 수리 효과’는 보류한다. 관측 공백·조건 차이·실제 수행 여부가 함께 남아 있다. **활용:** 계획 기간은 운전·수집 기록을 확인할 맥락으로 사용한다. 고장 학습 정답이나 안정화 완료 시각으로 사용하지 않는다.

<a id="cross-data"></a>
## 10. 서로 다른 CSV 사이의 관계는 어디까지 볼 수 있나?
**Reason:** 센서 상관뿐 아니라 조업·정비·교체 기록 사이의 관계를 보되, 서로 포함된 집계의 당연한 상관을 구분해야 한다.
**How:** CR1 조업, CAL01 정비, CAL01 교체 작업, 실제 센서 관측시간을 동일 12개월 축에 맞춘다.
**What / When:** 이벤트는 EVT_DT 월, 정비는 DETC_DT 월, 교체는 연결 END_DT 월이다. 서로 다른 시점 정의 때문에 같은 달의 동행도 인과·선후관계가 아니다. 센서 없는 달의 센서 평균은 0으로 채우지 않는다.

In [ ]:
monthly=pd.DataFrame(index=months)
monthly["CR1조업건수"]=cr1_event.groupby(cr1_event.EVT_DT.dt.to_period("M")).size().reindex(months,fill_value=0)
monthly["CAL정비건수"]=cal_maint.groupby(cal_maint.DETC_DT.dt.to_period("M")).size().reindex(months,fill_value=0)
monthly["CAL교체작업수"]=cal_parts.groupby(cal_parts.END_DT.dt.to_period("M")).WO_NO.nunique().reindex(months,fill_value=0)
good_dt=temp.dt_s.where(temp.segment_id.eq(temp.segment_id.shift()),0)
monthly["센서관측지지_시간"]=good_dt.groupby(temp.MEAS_DT.dt.to_period("M")).sum().reindex(months,fill_value=0)/3600
month_corr=monthly.corr()
fig,axes=plt.subplots(2,1,figsize=(14,9))
for c,col in [("CR1조업건수","#398557"),("CAL정비건수","#777777"),("CAL교체작업수","#9467BD")]:
    axes[0].plot(months.astype(str),monthly[c],marker="o",label=c,color=col)
axes[0].set(title="월별 기록 동행 / 서로 다른 사건 시각 정의",xlabel="2024년 월",ylabel="기록 [건]")
axes[0].legend()
sns.heatmap(month_corr,ax=axes[1],annot=True,fmt=".2f",vmin=-1,vmax=1,cmap="coolwarm",cbar_kws={"label":"12개월 Pearson r [무차원]"})
axes[1].set(title="월별 집계 관계: 고장 예측 성능이 아님")
plot(fig)
table(monthly.rename_axis("월"))
cross_pairs=[]
for i,a in enumerate(monthly):
    for b in monthly.columns[i+1:]:
        leave_one=[monthly.drop(monthly.index[k])[[a,b]].corr().loc[a,b] for k in range(len(monthly))]
        cross_pairs.append({"항목1":a,"항목2":b,"월수":len(monthly),"r":month_corr.loc[a,b],"한달제외_r최소":min(leave_one),"한달제외_r최대":max(leave_one)})
cross_summary=pd.DataFrame(cross_pairs)
table(cross_summary)


**Result:** CR1 조업 건수–CAL01 정비 건수의 월별 r은 **-0.110**, CAL01 정비–교체 작업수는 **-0.025**이다. 유효 관측 단위는 센서 34만 행이 아니라 **12개월**이다. 한 달씩 제외한 r 범위를 함께 보고 특정 월 의존 여부를 확인한다.

**가설 판단:** 교체는 정비 작업 일부이고 기준 월도 다르므로 두 건수의 관계를 독립 원인으로 해석하지 않는다. 센서 관측시간은 추출·수집 범위이며 가동시간이 아니다. **활용:** 이 결과는 기록 범위와 추가 조사 월을 정하는 참고자료다. 월별 집계를 초단위 행에 복제해 고장 예측 근거로 삼지 않는다.

## 11. 출력 0과 온도 반응
**Reason:** OP=0을 곧바로 목표온도 도달·정지·고장으로 해석할 수 있는지 검토한다.
**How:** OP-Z1~Z8의 LOT별 0 비율과 연속 구간을 계산한다. 결측·LOT·10초 초과 공백에서 연속 기록을 끊는다.
**What / When:** 0 비율의 분모는 OP 유효행, 지속시간은 처음~마지막으로 0을 관측한 시각 차이[초]다. 한 행만 0이면 0초이며 실제 짧은 지속시간까지 안다는 뜻은 아니다.

In [ ]:
zero_rows=[]; zero_runs=[]
for zone in range(1,9):
    op=f"OP-Z{zone}"; tc=f"TC-Z{zone}"
    for lot,rows in temp.groupby("LOT_NO"):
        zero_rows.append({"LOT":lot,"존":zone,"유효행":rows[op].count(),"0행":rows[op].eq(0).sum(),"0비율_%":rows[op].eq(0).sum()/rows[op].count()*100})
    is_zero=temp[op].eq(0)
    run_id=(is_zero.ne(is_zero.shift())|temp.segment_id.ne(temp.segment_id.shift())).cumsum()
    for _,rows in temp[is_zero].groupby(run_id[is_zero]):
        zero_runs.append({"LOT":int(rows.LOT_NO.iloc[0]),"존":zone,"시작":rows.MEAS_DT.min(),"끝":rows.MEAS_DT.max(),
            "관측지속_초":(rows.MEAS_DT.max()-rows.MEAS_DT.min()).total_seconds(),"0행":len(rows),
            "온도최소":rows[tc].min(),"온도최대":rows[tc].max()})
zero_stats=pd.DataFrame(zero_rows); zero_runs=pd.DataFrame(zero_runs)
fig,ax=plt.subplots(figsize=(13,6))
sns.heatmap(zero_stats.pivot(index="LOT",columns="존",values="0비율_%"),ax=ax,annot=True,fmt=".2f",cmap="YlOrRd",vmin=0,cbar_kws={"label":"출력 유효행 중 0 [%]"})
ax.set(title="LOT × 출력 존의 0 비율",xlabel="출력 존",ylabel="LOT 번호")
plot(fig)
table(zero_runs.sort_values("관측지속_초",ascending=False).head(15))
longest=zero_runs[zero_runs.존.eq(3)].sort_values("관측지속_초").iloc[-1]
view=temp[temp.LOT_NO.eq(longest.LOT)&temp.MEAS_DT.between(longest.시작-pd.Timedelta(minutes=5),longest.끝+pd.Timedelta(minutes=5))]
fig,axes=plt.subplots(2,1,figsize=(13,6),sharex=True)
for _,seg in view.groupby("segment_id"):
    x=(seg.MEAS_DT-longest.시작).dt.total_seconds()/60
    axes[0].plot(x,seg["OP-Z3"],color=SENSOR_COLORS["OP-Z3"],label="OP-Z3")
    for s in ["TC-Z3","TC-Z4"]: axes[1].plot(x,seg[s],color=SENSOR_COLORS[s],label=s)
for ax in axes:
    ax.axvspan(0,longest.관측지속_초/60,color="#F2C14E",alpha=.3,label="출력 0 관측 구간")
    legend_once(ax)
axes[0].set(title=f"LOT {int(longest.LOT)} / Z3 최장 0 구간",ylabel="OP-Z3 [%]")
axes[1].set(xlabel="0 관측 시작 기준 [분]",ylabel="온도 [°C]")
plot(fig)
table(pd.DataFrame([longest]))


**Result:** Z3 최장 0 기록은 **LOT 240061 / 396초**, 해당 구간 TC-Z3 범위는 **861.519~870.704°C**다. **출력이 0이어도 온도는 한 값에 고정되지 않는다.**

**가설 판단:** 목표도달 여부는 SP가 없어서 미확정, 고장 여부는 모드·실제 사건 확인이 없어 미확정이다. **활용:** 0 여부·관측 지속초·동시 온도 변화는 상태 특징으로 남기고, 고장 라벨로 쓰지 않는다. 시작부터 0인 구간은 더 이전부터 0일 수 있다.

<a id="features"></a>
## 12. 분석 근거를 과거 파생변수로 변환
**Reason:** 그림에서 본 수준·흔들림·제어·존 차이를 모델이 읽을 수 있는 값으로 만든다.
**How:** 각 LOT·연속구간 안에서 뒤쪽 60·300초 창의 평균·표준편차와 실제 초당 변화율을 계산한다. 전체 창 길이가 확보되고 예상 1Hz 관측의 80% 이상일 때만 창 특징을 유효하게 둔다. 보간·미래값 대체는 하지 않는다.
**What / When:** 60초는 짧은 변화, 300초는 더 긴 상태를 비교하기 위한 탐색 창이며 최적 창으로 증명된 값이 아니다. 현재 시점까지의 값만 포함한다.

**왜 34개인가?** 선행 분석의 TC-Z3~Z6·OP-Z3~Z4 중심 후보를 재현하는 기본 묶음이다. 전체 22신호 중 최적 6신호를 선정한 결과가 아니다. 이번에 강한 관계가 확인된 **TC-Z5–OP-Z5, TC-Z6–OP-Z6, TC-Z7–OP-Z7**은 후속 비교에서 추가할 후보다. CP는 정의·가용시각 확인 전까지 별도로 보류한다. CLN-IN·기타 존의 약한 동시 상관도 예측 가치가 없다는 증거는 아니다.

In [ ]:
SENSORS=["TC-Z3","TC-Z4","TC-Z5","TC-Z6","OP-Z3","OP-Z4"]
feature_chunks=[]
for seg,rows in temp.groupby("segment_id"):
    values=rows.set_index("MEAS_DT")
    f=values[["LOT_NO","segment_id"]].copy()
    elapsed=(values.index-values.index[0]).total_seconds()
    for s in SENSORS:
        f[s+"_rate_s"]=values[s].diff()/values.index.to_series().diff().dt.total_seconds()
        for seconds in [60,300]:
            roll=values[s].rolling(f"{seconds}s",min_periods=int(seconds*.8))
            ready=elapsed>=seconds
            f[s+f"_mean_{seconds}s"]=roll.mean().where(ready)
            f[s+f"_std_{seconds}s"]=roll.std().where(ready)
    f["tc_z3_z4_diff"]=values["TC-Z3"]-values["TC-Z4"]
    zones=values[["TC-Z3","TC-Z4","TC-Z5","TC-Z6"]]
    f["tc_z3_z6_spread"]=(zones.max(axis=1)-zones.min(axis=1)).where(zones.notna().all(axis=1))
    zero=values["OP-Z3"].eq(0)
    run=zero.ne(zero.shift()).cumsum()
    starts=values.index.to_series().groupby(run).transform("min")
    f["op_z3_zero"]=zero.astype(float).where(values["OP-Z3"].notna())
    f["op_z3_zero_seconds"]=(values.index.to_series()-starts).dt.total_seconds().where(zero,0).where(values["OP-Z3"].notna())
    feature_chunks.append(f.reset_index())
features=pd.concat(feature_chunks,ignore_index=True)
feature_cols=features.columns.drop(["MEAS_DT","LOT_NO","segment_id"]).tolist()
assert len(feature_cols)==34
assert len(features)==len(temp)
assert features[["LOT_NO","MEAS_DT"]].equals(temp[["LOT_NO","MEAS_DT"]])
features["model_ready"]=features[feature_cols].notna().all(axis=1)
feature_rows=[]
for c in feature_cols:
    base=c.split("_")[0]
    unit="무차원"
    if c.startswith("TC") or c.startswith("tc_"): unit="°C/초" if c.endswith("rate_s") else "°C"
    elif c.startswith("OP"): unit="%p/초" if c.endswith("rate_s") else "%p" if "_std_" in c else "%"
    else: unit="0/1" if c=="op_z3_zero" else "초"
    feature_rows.append({"파생변수":c,"단위":unit,"유효행":features[c].count(),"유효_%":features[c].notna().mean()*100,"고유값수":features[c].nunique(),"최소":features[c].min(),"중앙":features[c].median(),"최대":features[c].max()})
feature_dictionary=pd.DataFrame(feature_rows)
table(feature_dictionary)
case=features[features.LOT_NO.eq(240061)]
raw_case=temp[temp.LOT_NO.eq(240061)]
x=(case.MEAS_DT-case.MEAS_DT.min()).dt.total_seconds()/60
fig,axes=plt.subplots(3,1,figsize=(14,9),sharex=True)
axes[0].plot(x,raw_case["TC-Z3"],color=SENSOR_COLORS["TC-Z3"],lw=.6,label="TC-Z3 원본")
for sec,color in [(60,"#9467BD"),(300,"#8C564B")]:
    axes[0].plot(x,case[f"TC-Z3_mean_{sec}s"],color=color,label=f"과거 {sec}초 평균")
    axes[1].plot(x,case[f"TC-Z3_std_{sec}s"],color=color,label=f"과거 {sec}초 표준편차")
axes[2].plot(x,case.tc_z3_z4_diff,color="#17BECF",label="Z3-Z4")
axes[2].plot(x,case.tc_z3_z6_spread,color="#E377C2",label="Z3~Z6 최대-최소")
for ax in axes: ax.legend()
axes[0].set(title="LOT 240061: 원본 → 과거 특징",ylabel="온도 [°C]")
axes[1].set(ylabel="온도 표준편차 [°C]")
axes[2].set(ylabel="온도 차이 [°C]",xlabel="LOT 시작 후 [분]")
plot(fig)


**Result:** **34개** 후보를 만들었고, 모두 계산 조건을 충족한 행은 **340,235 / 345,390행**이다. 표의 유효율은 계산 가능성이지 정상 비율이 아니다.

| 특징 | 위 시각화에서 확인하는 현상 | 필요한 이유 |
|---|---|---|
| 60·300초 평균 | 짧고 긴 온도·출력 수준 변화 | 현재 값 하나와 시간 상태를 구분 |
| 60·300초 표준편차 | 같은 수준에서도 흔들림이 달라짐 | 불안정성 후보를 수치화 |
| 실제 초당 변화율 | 빠른 상승·하강 | 수집 간격 차이를 보정 |
| Z3−Z4 / Z3~Z6 범위 | 센서가 함께 움직여도 남는 차이 | 공통 변화와 존 불균형을 분리 |
| OP-Z3 0 여부·지속초 | 출력 차단의 상태·길이 | 모드 정보 확보 후 정상·비정상 차단 비교 |

**결론:** 특징을 계산했다는 것과 좋은 예측 변수를 선정했다는 것은 다르다. 이 05번에서는 해석 가능한 후보를 준비했으며, 고장 성능이나 최적 변수 조합을 확인한 것은 아니다.

### 12-1. 정비·교체 이후 경과시간은 열화 지표인가?
**Reason:** 정비·교체 기록을 센서와 시간으로 연결했을 때 의미 있는 변화가 생기는지 확인한다.
**How:** 현재 시각 이전에 **완료된** CAL01 정비와 품목별 교체 기록만 backward 연결한다. 최초 기록 이전은 결측으로 유지한다. 초→시간으로 환산하며 운전시간으로 부르지 않는다.
**What / When:** 나이처럼 증가하는 달력 경과시간이며, 실제 기록 등록이 늦었다면 당시 사용할 수 없을 수 있다. 온라인 활용 전 등록·가용시각 확인이 필요하다.

In [ ]:
context=temp[["LOT_NO","MEAS_DT","elapsed_min"]].copy()
def hours_after(times):
    right=pd.DataFrame({"완료시각":pd.to_datetime(times).drop_duplicates().sort_values()})
    joined=pd.merge_asof(context.sort_values("MEAS_DT"),right,left_on="MEAS_DT",right_on="완료시각",direction="backward")
    assert (joined.완료시각.dropna()<=joined.loc[joined.완료시각.notna(),"MEAS_DT"]).all()
    return (joined.MEAS_DT-joined.완료시각).dt.total_seconds().to_numpy()/3600
assert context.MEAS_DT.is_monotonic_increasing
context["정비후_시간"]=hours_after(cal_maint.END_DT)
for p in part_order:
    context[p+"_교체후_시간"]=hours_after(cal_parts.loc[cal_parts.PART_CD.eq(p),"END_DT"])
context_cols=context.columns.drop(["LOT_NO","MEAS_DT","elapsed_min"]).tolist()
fig,axes=plt.subplots(4,1,figsize=(14,10),sharex=True)
for ax,c in zip(axes,context_cols):
    for lot,rows in context.groupby("LOT_NO"):
        for _,seg in rows.groupby(temp.loc[rows.index,"segment_id"]):
            ax.plot(seg.MEAS_DT,seg[c],color=LOT_COLORS[lot],label=str(lot))
    ax.set_ylabel(c+"\n[달력 시간]")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
axes[0].set_title("마지막으로 기록된 완료 이후 경과시간 / 미확인 구간은 연결하지 않음")
legend_once(axes[0],ncol=7,fontsize=7,title="LOT")
axes[-1].set_xlabel("실제 관측 날짜")
plot(fig)
context_summary=context[context_cols].agg(["count","min","median","max"]).T
table(context_summary)
age_rows=[]
for lot,r in context.groupby("LOT_NO"):
    for c in context_cols:
        v=r[["elapsed_min",c]].dropna()
        corr=v.elapsed_min.corr(v[c]) if len(v)>=100 and v[c].nunique()>1 else np.nan
        age_rows.append({"LOT":lot,"지표":c,"유효행":len(v),"LOT경과시간과_r":corr,"리셋수":int(r[c].diff().lt(0).sum())})
age_audit=pd.DataFrame(age_rows)
table(age_audit)


**Result:** 정비 이후 경과시간이 연결된 센서 행은 **304,988행**이다. 표의 교체 경과시간은 품목별 첫 기록 이후만 존재한다. LOT 내부에서 리셋이 없으면 시간과 함께 거의 직선적으로 증가한다.

**가설 판단:** ‘오래 지났을수록 열화됐다’는 결론을 낼 수 없다. 경과시간–온도 관계는 같은 LOT의 공통 시간 추세·강종·운전 상태 때문일 수 있다. **활용:** 사건 맥락·시간 상태 후보로 보존하되, 단독 수명 지표로 채택하지 않는다. 실물 부품 식별·장착 위치·실가동시간·실제 고장과 연결해 별도 검증한다.

## 13. 감지 이전 관측이 있는 8개 작업의 사례
**Reason:** ‘예지보전 가능’이라는 주장을 직접 확인하려면 감지 전 신호가 있어야 한다.
**How:** 44작업 중 **감지 전 24시간에 센서가 한 행 이상 있는 모든 8작업**을 아래에 표시한다. 직전 1시간에 관측이 있는 2건은 확대하고 나머지 6건은 24시간 전체 창으로 관측 공백까지 보여준다. 사건을 고장으로 단정하지 않는다.
**What / When:** 0분은 DETC_DT, 음수 분은 그 이전이다. 급변·변동성·존 차이의 시작이 감지보다 앞서는지 보되, 정상 대조 구간과 실제 고장 확인이 없으므로 전조로 확정하지 않는다.

### WO-2024-01859 · 원본 정비유형 TBM
**목적:** 감지 전 신호가 실제로 얼마나 존재하고, TC·OP·CP·과거 변동성이 함께 움직이는지 확인한다. **방법:** 감지 전 60분 창으로 제한하고 센서·특징을 같은 상대시간에 표시한다. 표의 첫 비교창은 감지 60~50분 전, 마지막은 10~0분 전이다.

In [ ]:
job="WO-2024-01859"
w=cal_maint.set_index("WO_NO").loc[job]
pre=temp[temp.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=1),w.DETC_DT,inclusive="left")]
pre_f=features[features.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=1),w.DETC_DT,inclusive="left")]
fig,axes=plt.subplots(4,1,figsize=(14,11),sharex=True)
for _,seg in pre.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/60
    for s in ["TC-Z3","TC-Z4"]: axes[0].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    axes[2].plot(x,seg["CP-Z4"],color=SENSOR_COLORS["CP-Z4"],label="CP-Z4")
for _,seg in pre_f.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/60
    axes[3].plot(x,seg["TC-Z3_std_60s"],color="#9467BD",label="Z3 과거60초 표준편차")
for ax in axes:
    ax.axvline(0,color=WORK_COLORS[job],linestyle="--",label="감지 기록 시점")
    ax.set_xlim(-60,1)
    legend_once(ax,fontsize=8,ncol=3)
axes[0].set(title=f"{job} / {w.MNT_TYPE} / 감지 {w.DETC_DT} / 왼쪽 빈 곳은 미관측",ylabel="온도 [°C]")
axes[1].set_ylabel("출력 [%]"); axes[2].set_ylabel("CP-Z4 [무차원]")
axes[3].set(ylabel="온도 표준편차 [°C]",xlabel="감지 시각 기준 [분]")
plot(fig)
pre_stats=[]
for s in signals:
    early=pre[pre.MEAS_DT<w.DETC_DT-pd.Timedelta(minutes=50)][s]
    late=pre[pre.MEAS_DT>=w.DETC_DT-pd.Timedelta(minutes=10)][s]
    pre_stats.append({"센서":s,"단위":units[s],"전체유효행":pre[s].count(),"최소":pre[s].min(),"최대":pre[s].max(),
        "60~50분전평균":early.mean(),"10~0분전평균":late.mean(),"차이":late.mean()-early.mean(),"먼창유효행":early.count(),"직전창유효행":late.count()})
pre_stats=pd.DataFrame(pre_stats)
table(pre_stats)
record=work_coverage.set_index("WO_NO").loc[job]
pre_z3=pre_stats.set_index("센서").loc["TC-Z3"]


**Result:** 감지 전 1시간에 **290행**, 관측지지시간 비율 **8.31%**가 있다. TC-Z3 관측 범위는 **812.474~822.935°C**다. 먼 창·직전 창 유효행은 **0 / 290**이며, 어느 쪽이 0이면 전후 차이를 평가하지 않는다.

**가설 판단:** 이 자료는 **TBM 작업의 감지 이전 관측**이다. ‘변동이 보인다’와 ‘실제 고장을 미리 예측했다’는 다른 주장이다. 목표값·고장 확인·정상 대조 구간·경보 임곗값이 없으므로 고장 전조의 재현율·정밀도·선행시간은 아직 산출할 수 없다.

**활용:** 위 표의 TC·OP·CP 변화 및 표준편차가 실제 이상 증상과 맞는지 작업 상세·현장 기록으로 확인한다. 경보 기준은 별도 시간순 검증 구간에서 정하고 이 사례에 맞춰 사후 조정하지 않는다.

### WO-2024-02105 · 원본 정비유형 CBM
**목적:** 감지 전 신호가 실제로 얼마나 존재하고, TC·OP·CP·과거 변동성이 함께 움직이는지 확인한다. **방법:** 감지 전 60분 창으로 제한하고 센서·특징을 같은 상대시간에 표시한다. 표의 첫 비교창은 감지 60~50분 전, 마지막은 10~0분 전이다.

In [ ]:
job="WO-2024-02105"
w=cal_maint.set_index("WO_NO").loc[job]
pre=temp[temp.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=1),w.DETC_DT,inclusive="left")]
pre_f=features[features.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=1),w.DETC_DT,inclusive="left")]
fig,axes=plt.subplots(4,1,figsize=(14,11),sharex=True)
for _,seg in pre.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/60
    for s in ["TC-Z3","TC-Z4"]: axes[0].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    axes[2].plot(x,seg["CP-Z4"],color=SENSOR_COLORS["CP-Z4"],label="CP-Z4")
for _,seg in pre_f.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/60
    axes[3].plot(x,seg["TC-Z3_std_60s"],color="#9467BD",label="Z3 과거60초 표준편차")
for ax in axes:
    ax.axvline(0,color=WORK_COLORS[job],linestyle="--",label="감지 기록 시점")
    ax.set_xlim(-60,1)
    legend_once(ax,fontsize=8,ncol=3)
axes[0].set(title=f"{job} / {w.MNT_TYPE} / 감지 {w.DETC_DT} / 왼쪽 빈 곳은 미관측",ylabel="온도 [°C]")
axes[1].set_ylabel("출력 [%]"); axes[2].set_ylabel("CP-Z4 [무차원]")
axes[3].set(ylabel="온도 표준편차 [°C]",xlabel="감지 시각 기준 [분]")
plot(fig)
pre_stats=[]
for s in signals:
    early=pre[pre.MEAS_DT<w.DETC_DT-pd.Timedelta(minutes=50)][s]
    late=pre[pre.MEAS_DT>=w.DETC_DT-pd.Timedelta(minutes=10)][s]
    pre_stats.append({"센서":s,"단위":units[s],"전체유효행":pre[s].count(),"최소":pre[s].min(),"최대":pre[s].max(),
        "60~50분전평균":early.mean(),"10~0분전평균":late.mean(),"차이":late.mean()-early.mean(),"먼창유효행":early.count(),"직전창유효행":late.count()})
pre_stats=pd.DataFrame(pre_stats)
table(pre_stats)
record=work_coverage.set_index("WO_NO").loc[job]
pre_z3=pre_stats.set_index("센서").loc["TC-Z3"]


**Result:** 감지 전 1시간에 **3,309행**, 관측지지시간 비율 **94.97%**가 있다. TC-Z3 관측 범위는 **850.130~872.524°C**다. 먼 창·직전 창 유효행은 **407 / 582**이며, 어느 쪽이 0이면 전후 차이를 평가하지 않는다.

**가설 판단:** 이 자료는 **CBM 작업의 감지 이전 관측**이다. ‘변동이 보인다’와 ‘실제 고장을 미리 예측했다’는 다른 주장이다. 목표값·고장 확인·정상 대조 구간·경보 임곗값이 없으므로 고장 전조의 재현율·정밀도·선행시간은 아직 산출할 수 없다.

**활용:** 위 표의 TC·OP·CP 변화 및 표준편차가 실제 이상 증상과 맞는지 작업 상세·현장 기록으로 확인한다. 경보 기준은 별도 시간순 검증 구간에서 정하고 이 사례에 맞춰 사후 조정하지 않는다.

### 13-추가. 직전 1시간에는 없지만 24시간 안에 관측이 있는 모든 작업
**목적:** 직전 센서 부족을 전체 전조 자료 부재로 오해하지 않도록 더 긴 창을 확인한다. 아래는 위 2건을 제외한 6건 전체다. **방법:** 음수 시간 축을 -24~0시간으로 고정하고 관측 없는 끝 구간도 빈 상태로 남긴다. **판단:** 마지막 관측부터 감지까지의 공백과 정비유형을 함께 본다. BM·CBM·TBM 기록 자체는 검증된 고장 라벨이 아니다.

### WO-2024-00627 · CBM · 감지 전 24시간
**Reason:** 더 긴 선행시간에서 조사 가능한 센서가 있는 작업이다. **How / What:** 모든 22신호의 원본 범위·평균·유효행을 표로 확인하고 TC·OP·CP·변동성은 같은 시간축에 표시한다. **When:** 0시간은 감지 시각, 빈 곳은 실제 미관측이다.

In [ ]:
job="WO-2024-00627"
w=cal_maint.set_index("WO_NO").loc[job]
pre=temp[temp.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
pre_f=features[features.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
record=work_coverage.set_index("WO_NO").loc[job]
last_gap=(w.DETC_DT-pre.MEAS_DT.max()).total_seconds()/3600
fig,axes=plt.subplots(4,1,figsize=(14,10),sharex=True)
for _,seg in pre.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    for s in ["TC-Z3","TC-Z4"]: axes[0].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    axes[2].plot(x,seg["CP-Z4"],color=SENSOR_COLORS["CP-Z4"],label="CP-Z4")
for _,seg in pre_f.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    axes[3].plot(x,seg["TC-Z3_std_60s"],color="#9467BD",label="Z3 과거60초 표준편차")
for ax in axes:
    ax.axvline(0,color=WORK_COLORS[job],ls="--",label="감지 기록")
    ax.set_xlim(-24,.2)
    legend_once(ax,ncol=3,fontsize=8)
axes[0].set(title=f"{job} / {w.MNT_TYPE} / 마지막 관측→감지 공백 {last_gap:.3f}시간",ylabel="온도 [°C]")
axes[1].set_ylabel("출력 [%]"); axes[2].set_ylabel("CP-Z4 [무차원]")
axes[3].set(xlabel="감지 시각 기준 [시간]",ylabel="온도 표준편차 [°C]")
plot(fig)
pre_summary=pre[signals].agg(["count","min","mean","max"]).T.rename(columns={"count":"유효행","min":"최소","mean":"평균","max":"최대"})
pre_summary.insert(0,"단위",[units[s] for s in pre_summary.index])
table(pre_summary)


**Result:** 35,348행, 24시간 창 중 관측지지시간 **42.45%**, 마지막 관측→감지 공백 **5.611시간**이다. 관측 LOT는 **240037**, 감지 기록은 **2024-04-14 15:04:00**다.

**해석:** 선과 감지선 사이가 비어 있으므로 그 구간의 센서 변화는 알 수 없다. 이 CBM 작업의 이전 온도·출력·CP 상태를 확인할 사례는 확보했지만 ‘언제부터 고장 징후가 시작됐다’고 확정할 수 없다. **활용:** 실제 고장 확인 후 긴 선행시간 후보를 검토하고, 빈 구간을 보간해서 전조를 만들어내지 않는다.

### WO-2024-00717 · BM · 감지 전 24시간
**Reason:** 더 긴 선행시간에서 조사 가능한 센서가 있는 작업이다. **How / What:** 모든 22신호의 원본 범위·평균·유효행을 표로 확인하고 TC·OP·CP·변동성은 같은 시간축에 표시한다. **When:** 0시간은 감지 시각, 빈 곳은 실제 미관측이다.

In [ ]:
job="WO-2024-00717"
w=cal_maint.set_index("WO_NO").loc[job]
pre=temp[temp.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
pre_f=features[features.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
record=work_coverage.set_index("WO_NO").loc[job]
last_gap=(w.DETC_DT-pre.MEAS_DT.max()).total_seconds()/3600
fig,axes=plt.subplots(4,1,figsize=(14,10),sharex=True)
for _,seg in pre.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    for s in ["TC-Z3","TC-Z4"]: axes[0].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    axes[2].plot(x,seg["CP-Z4"],color=SENSOR_COLORS["CP-Z4"],label="CP-Z4")
for _,seg in pre_f.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    axes[3].plot(x,seg["TC-Z3_std_60s"],color="#9467BD",label="Z3 과거60초 표준편차")
for ax in axes:
    ax.axvline(0,color=WORK_COLORS[job],ls="--",label="감지 기록")
    ax.set_xlim(-24,.2)
    legend_once(ax,ncol=3,fontsize=8)
axes[0].set(title=f"{job} / {w.MNT_TYPE} / 마지막 관측→감지 공백 {last_gap:.3f}시간",ylabel="온도 [°C]")
axes[1].set_ylabel("출력 [%]"); axes[2].set_ylabel("CP-Z4 [무차원]")
axes[3].set(xlabel="감지 시각 기준 [시간]",ylabel="온도 표준편차 [°C]")
plot(fig)
pre_summary=pre[signals].agg(["count","min","mean","max"]).T.rename(columns={"count":"유효행","min":"최소","mean":"평균","max":"최대"})
pre_summary.insert(0,"단위",[units[s] for s in pre_summary.index])
table(pre_summary)


**Result:** 4,334행, 24시간 창 중 관측지지시간 **5.18%**, 마지막 관측→감지 공백 **3.689시간**이다. 관측 LOT는 **240044**, 감지 기록은 **2024-05-03 23:46:00**다.

**해석:** 선과 감지선 사이가 비어 있으므로 그 구간의 센서 변화는 알 수 없다. 이 BM 작업의 이전 온도·출력·CP 상태를 확인할 사례는 확보했지만 ‘언제부터 고장 징후가 시작됐다’고 확정할 수 없다. **활용:** 실제 고장 확인 후 긴 선행시간 후보를 검토하고, 빈 구간을 보간해서 전조를 만들어내지 않는다.

### WO-2024-00916 · TBM · 감지 전 24시간
**Reason:** 더 긴 선행시간에서 조사 가능한 센서가 있는 작업이다. **How / What:** 모든 22신호의 원본 범위·평균·유효행을 표로 확인하고 TC·OP·CP·변동성은 같은 시간축에 표시한다. **When:** 0시간은 감지 시각, 빈 곳은 실제 미관측이다.

In [ ]:
job="WO-2024-00916"
w=cal_maint.set_index("WO_NO").loc[job]
pre=temp[temp.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
pre_f=features[features.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
record=work_coverage.set_index("WO_NO").loc[job]
last_gap=(w.DETC_DT-pre.MEAS_DT.max()).total_seconds()/3600
fig,axes=plt.subplots(4,1,figsize=(14,10),sharex=True)
for _,seg in pre.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    for s in ["TC-Z3","TC-Z4"]: axes[0].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    axes[2].plot(x,seg["CP-Z4"],color=SENSOR_COLORS["CP-Z4"],label="CP-Z4")
for _,seg in pre_f.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    axes[3].plot(x,seg["TC-Z3_std_60s"],color="#9467BD",label="Z3 과거60초 표준편차")
for ax in axes:
    ax.axvline(0,color=WORK_COLORS[job],ls="--",label="감지 기록")
    ax.set_xlim(-24,.2)
    legend_once(ax,ncol=3,fontsize=8)
axes[0].set(title=f"{job} / {w.MNT_TYPE} / 마지막 관측→감지 공백 {last_gap:.3f}시간",ylabel="온도 [°C]")
axes[1].set_ylabel("출력 [%]"); axes[2].set_ylabel("CP-Z4 [무차원]")
axes[3].set(xlabel="감지 시각 기준 [시간]",ylabel="온도 표준편차 [°C]")
plot(fig)
pre_summary=pre[signals].agg(["count","min","mean","max"]).T.rename(columns={"count":"유효행","min":"최소","mean":"평균","max":"최대"})
pre_summary.insert(0,"단위",[units[s] for s in pre_summary.index])
table(pre_summary)


**Result:** 9,278행, 24시간 창 중 관측지지시간 **11.10%**, 마지막 관측→감지 공백 **21.335시간**이다. 관측 LOT는 **240058**, 감지 기록은 **2024-06-10 06:23:00**다.

**해석:** 선과 감지선 사이가 비어 있으므로 그 구간의 센서 변화는 알 수 없다. 이 TBM 작업의 이전 온도·출력·CP 상태를 확인할 사례는 확보했지만 ‘언제부터 고장 징후가 시작됐다’고 확정할 수 없다. **활용:** 실제 고장 확인 후 긴 선행시간 후보를 검토하고, 빈 구간을 보간해서 전조를 만들어내지 않는다.

### WO-2024-01094 · BM · 감지 전 24시간
**Reason:** 더 긴 선행시간에서 조사 가능한 센서가 있는 작업이다. **How / What:** 모든 22신호의 원본 범위·평균·유효행을 표로 확인하고 TC·OP·CP·변동성은 같은 시간축에 표시한다. **When:** 0시간은 감지 시각, 빈 곳은 실제 미관측이다.

In [ ]:
job="WO-2024-01094"
w=cal_maint.set_index("WO_NO").loc[job]
pre=temp[temp.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
pre_f=features[features.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
record=work_coverage.set_index("WO_NO").loc[job]
last_gap=(w.DETC_DT-pre.MEAS_DT.max()).total_seconds()/3600
fig,axes=plt.subplots(4,1,figsize=(14,10),sharex=True)
for _,seg in pre.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    for s in ["TC-Z3","TC-Z4"]: axes[0].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    axes[2].plot(x,seg["CP-Z4"],color=SENSOR_COLORS["CP-Z4"],label="CP-Z4")
for _,seg in pre_f.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    axes[3].plot(x,seg["TC-Z3_std_60s"],color="#9467BD",label="Z3 과거60초 표준편차")
for ax in axes:
    ax.axvline(0,color=WORK_COLORS[job],ls="--",label="감지 기록")
    ax.set_xlim(-24,.2)
    legend_once(ax,ncol=3,fontsize=8)
axes[0].set(title=f"{job} / {w.MNT_TYPE} / 마지막 관측→감지 공백 {last_gap:.3f}시간",ylabel="온도 [°C]")
axes[1].set_ylabel("출력 [%]"); axes[2].set_ylabel("CP-Z4 [무차원]")
axes[3].set(xlabel="감지 시각 기준 [시간]",ylabel="온도 표준편차 [°C]")
plot(fig)
pre_summary=pre[signals].agg(["count","min","mean","max"]).T.rename(columns={"count":"유효행","min":"최소","mean":"평균","max":"최대"})
pre_summary.insert(0,"단위",[units[s] for s in pre_summary.index])
table(pre_summary)


**Result:** 46,836행, 24시간 창 중 관측지지시간 **56.40%**, 마지막 관측→감지 공백 **2.324시간**이다. 관측 LOT는 **240069**, 감지 기록은 **2024-07-08 08:49:00**다.

**해석:** 선과 감지선 사이가 비어 있으므로 그 구간의 센서 변화는 알 수 없다. 이 BM 작업의 이전 온도·출력·CP 상태를 확인할 사례는 확보했지만 ‘언제부터 고장 징후가 시작됐다’고 확정할 수 없다. **활용:** 실제 고장 확인 후 긴 선행시간 후보를 검토하고, 빈 구간을 보간해서 전조를 만들어내지 않는다.

### WO-2024-01678 · CBM · 감지 전 24시간
**Reason:** 더 긴 선행시간에서 조사 가능한 센서가 있는 작업이다. **How / What:** 모든 22신호의 원본 범위·평균·유효행을 표로 확인하고 TC·OP·CP·변동성은 같은 시간축에 표시한다. **When:** 0시간은 감지 시각, 빈 곳은 실제 미관측이다.

In [ ]:
job="WO-2024-01678"
w=cal_maint.set_index("WO_NO").loc[job]
pre=temp[temp.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
pre_f=features[features.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
record=work_coverage.set_index("WO_NO").loc[job]
last_gap=(w.DETC_DT-pre.MEAS_DT.max()).total_seconds()/3600
fig,axes=plt.subplots(4,1,figsize=(14,10),sharex=True)
for _,seg in pre.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    for s in ["TC-Z3","TC-Z4"]: axes[0].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    axes[2].plot(x,seg["CP-Z4"],color=SENSOR_COLORS["CP-Z4"],label="CP-Z4")
for _,seg in pre_f.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    axes[3].plot(x,seg["TC-Z3_std_60s"],color="#9467BD",label="Z3 과거60초 표준편차")
for ax in axes:
    ax.axvline(0,color=WORK_COLORS[job],ls="--",label="감지 기록")
    ax.set_xlim(-24,.2)
    legend_once(ax,ncol=3,fontsize=8)
axes[0].set(title=f"{job} / {w.MNT_TYPE} / 마지막 관측→감지 공백 {last_gap:.3f}시간",ylabel="온도 [°C]")
axes[1].set_ylabel("출력 [%]"); axes[2].set_ylabel("CP-Z4 [무차원]")
axes[3].set(xlabel="감지 시각 기준 [시간]",ylabel="온도 표준편차 [°C]")
plot(fig)
pre_summary=pre[signals].agg(["count","min","mean","max"]).T.rename(columns={"count":"유효행","min":"최소","mean":"평균","max":"최대"})
pre_summary.insert(0,"단위",[units[s] for s in pre_summary.index])
table(pre_summary)


**Result:** 26,982행, 24시간 창 중 관측지지시간 **32.35%**, 마지막 관측→감지 공백 **3.985시간**이다. 관측 LOT는 **240108**, 감지 기록은 **2024-10-18 15:50:00**다.

**해석:** 선과 감지선 사이가 비어 있으므로 그 구간의 센서 변화는 알 수 없다. 이 CBM 작업의 이전 온도·출력·CP 상태를 확인할 사례는 확보했지만 ‘언제부터 고장 징후가 시작됐다’고 확정할 수 없다. **활용:** 실제 고장 확인 후 긴 선행시간 후보를 검토하고, 빈 구간을 보간해서 전조를 만들어내지 않는다.

### WO-2024-01922 · CBM · 감지 전 24시간
**Reason:** 더 긴 선행시간에서 조사 가능한 센서가 있는 작업이다. **How / What:** 모든 22신호의 원본 범위·평균·유효행을 표로 확인하고 TC·OP·CP·변동성은 같은 시간축에 표시한다. **When:** 0시간은 감지 시각, 빈 곳은 실제 미관측이다.

In [ ]:
job="WO-2024-01922"
w=cal_maint.set_index("WO_NO").loc[job]
pre=temp[temp.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
pre_f=features[features.MEAS_DT.between(w.DETC_DT-pd.Timedelta(hours=24),w.DETC_DT,inclusive="left")]
record=work_coverage.set_index("WO_NO").loc[job]
last_gap=(w.DETC_DT-pre.MEAS_DT.max()).total_seconds()/3600
fig,axes=plt.subplots(4,1,figsize=(14,10),sharex=True)
for _,seg in pre.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    for s in ["TC-Z3","TC-Z4"]: axes[0].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    for s in ["OP-Z3","OP-Z4"]: axes[1].plot(x,seg[s],color=SENSOR_COLORS[s],label=s,lw=.8)
    axes[2].plot(x,seg["CP-Z4"],color=SENSOR_COLORS["CP-Z4"],label="CP-Z4")
for _,seg in pre_f.groupby("segment_id"):
    x=(seg.MEAS_DT-w.DETC_DT).dt.total_seconds()/3600
    axes[3].plot(x,seg["TC-Z3_std_60s"],color="#9467BD",label="Z3 과거60초 표준편차")
for ax in axes:
    ax.axvline(0,color=WORK_COLORS[job],ls="--",label="감지 기록")
    ax.set_xlim(-24,.2)
    legend_once(ax,ncol=3,fontsize=8)
axes[0].set(title=f"{job} / {w.MNT_TYPE} / 마지막 관측→감지 공백 {last_gap:.3f}시간",ylabel="온도 [°C]")
axes[1].set_ylabel("출력 [%]"); axes[2].set_ylabel("CP-Z4 [무차원]")
axes[3].set(xlabel="감지 시각 기준 [시간]",ylabel="온도 표준편차 [°C]")
plot(fig)
pre_summary=pre[signals].agg(["count","min","mean","max"]).T.rename(columns={"count":"유효행","min":"최소","mean":"평균","max":"최대"})
pre_summary.insert(0,"단위",[units[s] for s in pre_summary.index])
table(pre_summary)


**Result:** 14,287행, 24시간 창 중 관측지지시간 **16.89%**, 마지막 관측→감지 공백 **19.947시간**이다. 관측 LOT는 **240121**, 감지 기록은 **2024-11-25 14:19:00**다.

**해석:** 선과 감지선 사이가 비어 있으므로 그 구간의 센서 변화는 알 수 없다. 이 CBM 작업의 이전 온도·출력·CP 상태를 확인할 사례는 확보했지만 ‘언제부터 고장 징후가 시작됐다’고 확정할 수 없다. 

**활용:** 실제 고장 확인 후 긴 선행시간 후보를 검토하고, 빈 구간을 보간해서 전조를 만들어내지 않는다.

<a id="final"></a>
## 14. 통합 연결 그림과 최종 결론
**Reason:** 어떤 파일이 무엇을 설명하고, 어느 연결에 실제 예지보전 근거가 있는지 한눈에 정리한다.
**How / What:** 아래 흐름도의 각 상자는 이미 계산한 범위다. 키 연결·시간 정렬·상관·고장 예측은 서로 다른 검증 단계이며 화살표가 인과를 뜻하지 않는다.
**When:** 2024년 관측·사건에 대한 사후 탐색. 새로운 기간의 예측 성능은 별도 검증 대상이다.

In [ ]:
pre1=work_coverage[work_coverage["감지전1h행"].gt(0)]
bm_pre1=int(pre1.유형.eq("BM").sum())
pre24=work_coverage[work_coverage["감지전24h행"].gt(0)]
bm_pre24=int(pre24.유형.eq("BM").sum())
fig,ax=plt.subplots(figsize=(16,8))
ax.set_xlim(0,10); ax.set_ylim(0,8); ax.axis("off")
boxes=[
(1.6,6.9,"온도·출력·CP\n"+f"{len(temp):,}행 / {len(spans)} LOT / {len(signals)}신호","#DCECF7"),
(5,6.9,"설비마스터\nEQP_TAG = CAL01\n설치·중요도·대상 확인","#E6E6E6"),
(8.4,6.9,"조업이벤트·계획수리\nCR1 조건·계획 기간\n변경 효과·수행 여부 미확정","#E6F0E6"),
(1.6,4.4,"시계열 관계\n"+f"OP-Z5–CP-Z4M r={best_pair['변화율_r중앙']:.3f}\nTC-Z4–OP-Z4 r={lag_median.loc[4,0]:.3f}\nLOT 변화율 중앙값 / CP 정의 미확인","#DCECF7"),
(5,4.4,"정비이력\n"+f"CAL01 {len(cal_maint)}작업\n감지·착수·완료 분리","#F5E9DC"),
(8.4,4.4,"부품교체 → 부품마스터\n"+f"{len(cal_parts)}행 / {cal_parts.WO_NO.nunique()}작업 / {cal_parts.PART_CD.nunique()}품목\n完了時刻".replace("完了時刻","완료시각 대리값·기준수명"),"#EEE4F5"),
(1.6,1.6,"과거 특징 후보\n"+f"{len(feature_cols)}개 / 유효 {features.model_ready.sum():,}행\n평균·변동·변화율·존차이·0상태","#DCECF7"),
(5,1.6,"현재 확보한 예지보전 근거\n"+f"감지 전1h {len(pre1)}작업 / BM {bm_pre1}건\n전24h {len(pre24)}작업 / BM {bm_pre24}건\n고장 정답·관측 공백 추가 확인","#FFF0C9"),
(8.4,1.6,"다음 검증\n실제 고장·모드·센서 위치 확보\n시간순 평가 → 오경보·선행시간","#E6F0E6")]
for x,y,label,color in boxes:
    ax.text(x,y,label,ha="center",va="center",fontsize=11,bbox={"boxstyle":"round,pad=.8","facecolor":color,"edgecolor":"#617182"})
for x1,y1,x2,y2 in [(2.9,6.9,3.7,6.9),(5,6.1,5,5.2),(6.3,4.4,7.1,4.4),(1.6,6.1,1.6,5.2),(1.6,3.6,1.6,2.4),(5,3.6,5,2.4),(2.9,1.6,3.7,1.6),(6.3,1.6,7.1,1.6),(7.1,6.9,6.3,6.9),(8.4,3.6,8.4,2.4)]:
    ax.annotate("",xy=(x2,y2),xytext=(x1,y1),arrowprops={"arrowstyle":"->","color":"#617182","lw":1.5})
ax.text(3.3,7.25,"파일명 → EQP_TAG",ha="center",fontsize=8)
ax.text(6.7,7.25,"PLANT_CD·시각",ha="center",fontsize=8)
ax.text(5.12,5.65,"EQP_TAG",fontsize=8)
ax.text(6.65,4.7,"WO_NO → PART_CD",ha="center",fontsize=8)
ax.set_title("7개 자료의 역할 → 확인한 근거 → 예지보전 검증으로 연결",fontsize=16,pad=20)
plot(fig)
validation=pd.DataFrame([
["센서 행 보존",len(features)==len(temp)],
["키 중복 없음",not features.duplicated(["LOT_NO","MEAS_DT"]).any()],
["세그먼트 첫 변화율은 결측",features.groupby("segment_id").head(1)[[c for c in feature_cols if c.endswith("_rate_s")]].isna().all().all()],
["유효행의 특징 결측 없음",features.loc[features.model_ready,feature_cols].notna().all().all()],
["특징 무한대 없음",not np.isinf(features[feature_cols].to_numpy()).any()],
["22신호 × 14 LOT 통계",len(sensor_stats)==len(signals)*len(spans)],
["교체 연결 행 보존",len(linked)==len(change)],
],columns=["검사","통과"])
assert validation.통과.all()
table(validation)


### 최종 결과: 확인한 근거와 아직 부족한 근거

| 질문 | 실제 결과 | 해석·활용 |
|---|---|---|
| 모든 자료를 연결했는가? | 7 CSV → 22신호·14 LOT·CAL01 44작업·교체 9행 | 정확한 설비·작업·부품 키와 시간으로 범위를 한정 |
| 어떤 센서 관계가 강한가? | 변화율 상관 최상위 **OP-Z5–CP-Z4M**, LOT 중앙 r **-0.960** | 전체 상관보다 LOT별 반복·실제 변화율·유효쌍을 함께 확인 |
| 출력과 온도는 어떤 관계인가? | Z3 동시 변화율 r **-0.837**, Z4 **-0.956** | 제어 관계의 후보 근거. OP=0을 목표도달·고장으로 해석하지 않음 |
| 강종·정비 효과를 증명했는가? | 강종 전후10분 양쪽 관측 **0건**; 정비 양쪽 관측 **2건**, 비중첩 **0건** | 비교조건·중첩 문제로 인과 효과 보류 |
| 교체 후 얼마나 사용했는가? | 품목별 이전 교체기록 간격 6개, 달력 경과시간 연결 | 실제 실물 수명·가동시간·RUL은 아직 알 수 없음 |
| 모델이 읽을 특징은 준비됐는가? | 과거 특징 **34개**, 계산 충족 **340,235행** | 센서 변화·흔들림·존 불균형·출력 상태의 후보 |
| 고장 전조 검증 근거는 충분한가? | 감지 전1시간 **2작업·BM 0건** / 전24시간 **8작업·BM 2건** | 관측 있는 작업의 유형부터 확인해야 함. 실제 고장 예측 성능을 주장할 근거는 아직 부족 |

**전조 탐색이 완전히 불가능하다는 뜻은 아니다.** 24시간 창에는 BM 기록 2건의 이전 센서가 있다. 다만 WO-2024-00717은 감지 **3.689시간**, WO-2024-01094는 **2.324시간** 전에 관측이 끝난다. 긴 선행시간 가설의 탐색 사례로 쓸 수 있지만 직전 변화·실제 고장 시점·반복성은 확인되지 않았다. **감지 시각과 고장 발생 시각도 같다고 가정하지 않는다.**

**왜 함께 필요한가?** 온도는 **현상**, 출력은 **제어 상태**, LOT·강종은 **비교 조건**, 설비는 **대상**, 정비·교체는 **사건 맥락**, 부품마스터는 **기준·조달 정보**, 계획수리는 **예정된 운영 맥락**을 제공한다. 이들을 분리하지 않고 섞으면 정상 조업 변화·정비 후 정보·관측 공백을 고장 전조로 오인할 수 있다.

**남긴 가설**
1. **TC 단독보다 TC·OP·존 차이의 결합이 상태 변화를 더 잘 설명한다.** → 새 시간순 LOT에서 단독/결합 모델을 같은 시점에 비교한다.
2. **출력 0의 지속시간과 온도 반응을 함께 보면 운전 상태를 구분할 수 있다.** → SP·운전 모드를 확보하고 상태별로 검증한다.
3. **일부 고장 전에 변동성·관계 변화가 반복된다.** → 실제 고장이 확인된 여러 사건과 정상 대조 기간이 필요하다. 현재 사례만으로 증명되지 않았다.
4. **교체 후 경과시간이 위험도에 정보를 더한다.** → 실물·장착 위치·가동시간을 확보하고 LOT·강종·시간 추세를 통제한 비교가 필요하다.

**예지보전으로 이어지는 다음 순서:**  
실제 고장·발생/감지·기록 가용시각 확인 → 사건 전 센서 확보 → 예측 선행시간 정의 → LOT·시간순 분할 → train에서만 전처리 학습 → validation에서 특징·경보 임곗값 선택 → 새 고장 사건에서 **재현율·정밀도·가동시간당 오경보·선행시간** 평가.

**최종 결론:** 이번에 확보한 것은 **‘어떤 신호와 이력을 왜 함께 봐야 하는지’에 대한 관측 근거와 과거 특징 후보**다. 고장 원인·잔여수명·예지보전 성능 향상을 증명한 것은 아니다. 특히 감지 직전 관측과 고장 정답 부족을 먼저 보완해야, 지금의 분석을 실제로 일찍 발견하고 덜 오경보하는 모델로 이어갈 수 있다.

### 결과를 확인하는 방법
이 노트북에서 위 목차·LOT별 그림·근거표·마크다운 해석을 읽는다. `sensor_stats`, `pair_summary`, `lag_results`, `work_coverage`, `features`, `feature_dictionary`, `context`에 계산 결과가 남는다. 모든 원본은 읽기만 한다. [06: 특징·임계값·모델 검증](06_timeseries_feature_story_jinny.ipynb#handoff)에서 이 분석의 가설을 이어 검증한다. [그림 목록](../figures/05_all_data_timeseries_jinny/README.md)에서 저장된 시각화를 확인한다.

원본 해시는 실행 변수 `source_hashes`로 확인할 수 있다. 원본이나 분석 조건을 바꾸면 마크다운 수치도 새 계산값과 대조해야 한다. 공개 커밋 전에는 저장소 규칙대로 실행 출력을 제거한다.